# Geostationary Satellites in Rubin Data — ASCENT as a Landolt Analog

**Goal.** The proposed **Landolt** mission will place an artificial calibration star
(a precisely-known flux source) in **geostationary orbit** as a photometric calibrator for
Rubin/LSST. Before that flies, we want to know: *how bright does a comparably-sized GEO
satellite appear in Rubin data naturally?*

**ASCENT** (NORAD **51287**, intl. designator 2021-118E) is a small geostationary
satellite of similar size to the proposed Landolt spacecraft. This notebook:

1. Loads the ASCENT TLE and propagates it with `skyfield` as seen from Cerro Pachón.
2. Establishes the geometry — where does a GEO satellite sit in Rubin's sky?
3. Cross-matches the satellite track against **LSSTCam exposure pointings** (ConsDB) on a
   given night to find exposures whose field of view contained ASCENT.
4. Outlines how to pull the image cutout and measure the natural brightness.

> **Caveat on TLE epoch.** TLE accuracy degrades away from its epoch (GEO: ~km/day along-track).
> For a *specific historical night*, fetch the TLE whose epoch is closest to that night
> (Space-Track GP history). The hardcoded TLE below is a recent snapshot — fine for a
> night near its epoch, and for establishing the broad geometry.

---

> ### ⚠️ Results up front
> **1. On the TLE-epoch night (late June 2026), ASCENT is NOT visible from Rubin.** Its
> sub-satellite point is at ~90° E — the far side of the Earth from Cerro Pachón — so its
> altitude is ≈ −60° (below the horizon). See Section 3a.
>
> **2. But it IS visible periodically.** ASCENT is a completed-mission **12U CubeSat** that
> is **no longer station-kept**: its mean motion (1.01223 rev/day) exceeds geostationary
> (1.00274), so it **drifts eastward ~3.4°/day and laps the GEO belt every ~105 days**. Each
> lap it spends ~3 weeks in Rubin's visibility window (sub-lon −152°…+10° E). Section 3b finds
> windows around **~2026-04-25→05-27** and **~2026-08-08→09-09** (indicative; refine with a
> Space-Track TLE near the target date). To catch ASCENT in LSST data, point the cross-match
> at a night inside one of these windows.
>
> **3. Similar-sized GEO analogs (Section 6).** ASCENT's 12U size is nearly unique in GEO —
> almost all station-kept GEO objects are large, bright comsats, *unlike* the faint Landolt
> source. ~126 GEO satellites sit above 30° altitude from Rubin on a given night, but for a
> true small/faint analog the best target is **ASCENT itself during a drift window**.

> ### ✅ Confirmed with Space-Track historical TLEs (Section 7)
> Using 715 epoch-accurate TLEs over 2025-06 → 2026-06 (nearest-epoch per night, TLE age < 1 day):
> - **Four visibility windows** in the past year (each ~34–35 nights, peak altitude ~59°):
>   **2025-06-13→07-17**, **2025-09-26→10-30**, **2026-01-09→02-11**, **2026-04-24→05-27**.
>   (Matches the ~105-day drift lap.)
> - **23 LSSTCam science exposures actually contained ASCENT** within the 1.75° FoV across
>   60 visible nights that had data. Closest approach **0.46°** (exposure 2026042600279).
>   Matched exposures are in `data/ascent_fov_hits.parquet`. Bands: mostly **y/z/i** (red —
>   consistent with a sunlit object low in a bright twilight-ish sky).
> - **So yes — Rubin observes ASCENT naturally.** Next: load these calexps via Butler, locate
>   the trail, measure flux + zeropoint → natural brightness vs. the Landolt target spec.

## 1. Setup

In [ ]:
# ── Load DP2 stack environment from repo-root .env (for the Butler in Section 8) ──
# Run this notebook with the lsst-scipipe-13.0.0 kernel; this .env adds the DRP repo
# index + PG credentials so the DP2 Butler resolves. Safe no-op if already set.
import os, sys, pathlib

ENV_FILE = pathlib.Path("../.env").resolve()
if ENV_FILE.exists():
    PATH_VARS = {"PYTHONPATH", "PATH", "LD_LIBRARY_PATH"}
    with open(ENV_FILE) as _f:
        for _line in _f:
            _line = _line.strip()
            if not _line or _line.startswith("#") or "=" not in _line:
                continue
            _k, _v = _line.split("=", 1)
            if _k in PATH_VARS:
                _ex = os.environ.get(_k, "")
                _new = [p for p in _v.split(":") if p and p not in _ex.split(":")]
                if _new:
                    os.environ[_k] = ":".join(_new) + ((":" + _ex) if _ex else "")
            else:
                os.environ.setdefault(_k, _v)
    print(f"Loaded DP2 stack env from {ENV_FILE}")
else:
    print(
        f"NOTE: {ENV_FILE} not found — Section 8 (DP2 Butler) will not connect. "
        "Sections 1-7 (ConsDB/TLE) still work."
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone

from skyfield.api import EarthSatellite, load, wgs84
from astropy.coordinates import SkyCoord
import astropy.units as u

import sqlalchemy

%matplotlib inline
pd.set_option("display.max_columns", 40)
print("imports ok")

In [ ]:
# ── Rubin Observatory site (Cerro Pachón, Chile) ──────────────────────────────
RUBIN_LAT = -30.244633  # deg
RUBIN_LON = -70.749417  # deg  (West)
RUBIN_ELEV = 2663.0  # m

ts = load.timescale()
rubin = wgs84.latlon(RUBIN_LAT, RUBIN_LON, elevation_m=RUBIN_ELEV)
print(f"Rubin site: lat={RUBIN_LAT:.4f}  lon={RUBIN_LON:.4f}  elev={RUBIN_ELEV} m")

In [ ]:
# ── ConsDB (PostgreSQL logical replica) ───────────────────────────────────────
CONSDB_HOST = "usdf-summitdb-logical-replica-svc.sdf.slac.stanford.edu"
CONSDB_DB = "exposurelog"
CONSDB_USER = "usdf"
PGPASS_FILE = "/sdf/home/s/stalder/.lsst/postgres-credentials.txt"


def load_pgpass(path, host, database, user):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split(":")
            if (
                len(parts) >= 5
                and parts[0] == host
                and parts[2] == database
                and parts[3] == user
            ):
                return ":".join(parts[4:])
    raise ValueError(f"No credentials for {user}@{host}/{database}")


db_pass = load_pgpass(PGPASS_FILE, CONSDB_HOST, CONSDB_DB, CONSDB_USER)
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{CONSDB_USER}:{db_pass}@{CONSDB_HOST}/{CONSDB_DB}",
    connect_args={"connect_timeout": 30},
)


def consdb_query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(sqlalchemy.text(sql), conn)


print("ConsDB engine ready.")

## 2. ASCENT TLE

NORAD 51287. The TLE below was fetched from CelesTrak (epoch ~day 179 of 2026). To pull a
TLE near a specific historical night, use Space-Track's GP history API (requires login) and
replace the two lines here.

In [ ]:
# ASCENT — NORAD 51287  (CelesTrak GP, epoch ~2026 day 179)
ASCENT_NAME = "ASCENT"
ASCENT_L1 = "1 51287U 21118E   26179.18883349 -.00000315  00000+0  00000+0 0  9993"
ASCENT_L2 = "2 51287   4.1993  77.1029 0005179 108.3962 251.3353  1.01223170 16431"

ascent = EarthSatellite(ASCENT_L1, ASCENT_L2, ASCENT_NAME, ts)
print(ascent)
print(f"Epoch (UTC): {ascent.epoch.utc_iso()}")
# Sanity: mean motion ~1.0 rev/day => geosynchronous
mm = float(ASCENT_L2[52:63])
print(f"Mean motion: {mm:.5f} rev/day  ->  period {24/mm:.2f} h  (GEO ~= 23.93 h)")

## 3. Where does ASCENT sit in Rubin's sky?

Propagate over a full UTC day and compute topocentric **Alt/Az** and **RA/Dec** as seen
from Rubin. A GEO satellite is nearly fixed in alt/az; from a southern-hemisphere site the
geostationary belt sits in the *northern* sky at modest altitude. We check that ASCENT is
above the horizon during local night before bothering with exposure cross-matching.

In [ ]:
# Choose a night to examine (UTC date). Pick one close to the TLE epoch.
NIGHT = "2026-06-27"  # UTC calendar date of the evening->morning observing window

# Sample the whole UTC day at 5-minute cadence
day0 = datetime.strptime(NIGHT, "%Y-%m-%d").replace(tzinfo=timezone.utc)
minutes = np.arange(0, 24 * 60, 5)
times = ts.utc(day0.year, day0.month, day0.day, 0, minutes)

topo = (ascent - rubin).at(times)
alt, az, dist = topo.altaz()
ra, dec, _ = topo.radec()

track = pd.DataFrame(
    {
        "utc": [t.utc_datetime() for t in times],
        "alt_deg": alt.degrees,
        "az_deg": az.degrees,
        "ra_deg": ra._degrees,
        "dec_deg": dec.degrees,
        "range_km": dist.km,
    }
)
print(f"ASCENT as seen from Rubin on {NIGHT}:")
print(f"  altitude:  {track.alt_deg.min():.2f} .. {track.alt_deg.max():.2f} deg")
print(f"  azimuth:   {track.az_deg.min():.2f} .. {track.az_deg.max():.2f} deg")
print(f"  RA:        {track.ra_deg.min():.3f} .. {track.ra_deg.max():.3f} deg")
print(f"  Dec:       {track.dec_deg.min():.3f} .. {track.dec_deg.max():.3f} deg")
print(f"  range:     {track.range_km.min():.0f} .. {track.range_km.max():.0f} km")
track.head()

In [ ]:
# Visualize the (small) diurnal loop in RA/Dec and the alt/az position
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc = axes[0].scatter(track.ra_deg, track.dec_deg, c=track.alt_deg, cmap="viridis", s=12)
axes[0].set_xlabel("RA [deg]")
axes[0].set_ylabel("Dec [deg]")
axes[0].set_title(f"ASCENT topocentric RA/Dec from Rubin ({NIGHT})")
plt.colorbar(sc, ax=axes[0], label="altitude [deg]")

axes[1].scatter(track.az_deg, track.alt_deg, c=track.alt_deg, cmap="viridis", s=12)
axes[1].axhline(0, color="r", lw=1, ls="--", label="horizon")
axes[1].set_xlabel("Azimuth [deg]")
axes[1].set_ylabel("Altitude [deg]")
axes[1].set_title("Alt/Az (GEO ~ fixed point)")
axes[1].legend()
plt.tight_layout()

## 3a. Visibility gate — can Rubin *ever* see this GEO satellite?

A geostationary satellite is parked at a fixed **sub-satellite longitude**. It is only
geometrically visible from a ground site whose longitude is within ~±81° of the satellite's
(beyond that the satellite is below the horizon, behind the Earth). This is a hard gate:
if it fails, *no night will ever work* and there is no point scanning exposures.

In [ ]:
# Sub-satellite longitude and Rubin visibility gate
mins = np.arange(0, 24 * 60, 30)
t_day = ts.utc(day0.year, day0.month, day0.day, 0, mins)
sp = wgs84.subpoint(ascent.at(t_day))
sat_lon = sp.longitude.degrees
sat_lat = sp.latitude.degrees

dlon = abs(((sat_lon.mean() - RUBIN_LON) + 180) % 360 - 180)
print(
    f"ASCENT sub-satellite longitude: {sat_lon.mean():.1f} deg E "
    f"(box {sat_lon.min():.1f}..{sat_lon.max():.1f})"
)
print(
    f"ASCENT sub-satellite latitude:  {sat_lat.min():.1f}..{sat_lat.max():.1f} deg "
    f"(inclination ~{float(ASCENT_L2[8:16]):.1f} deg)"
)
print(f"Rubin longitude:                {RUBIN_LON:.1f} deg E")
print(f"|Delta longitude|:              {dlon:.1f} deg  (visibility limit ~81 deg)")
print()
alt_all = (ascent - rubin).at(t_day).altaz()[0].degrees
if dlon < 81 and alt_all.max() > 0:
    print(f"==> VISIBLE: ASCENT reaches altitude {alt_all.max():.1f} deg from Rubin.")
else:
    print(
        f"==> NOT VISIBLE from Rubin: max altitude {alt_all.max():.1f} deg (always below horizon)."
    )
    print("    This TLE parks ASCENT on the far side of the Earth from Cerro Pachon.")
    print("    Rubin cannot observe it at this longitude on any night.")

## 3b. Was ASCENT *ever* visible? — it drifts

ASCENT's mean motion is **1.01223 rev/day** vs. true geostationary **1.00274**. That ~1%
excess means ASCENT is **not station-kept** — it is a completed-mission 12U CubeSat
(AFRL tech demo) **drifting eastward around the GEO belt at ~3.4°/day**, lapping every
**~105 days**. So its sub-satellite longitude sweeps through *all* longitudes over a lap,
including Rubin's visibility window (~−152° to +10° E).

Below we propagate ±120 days from the TLE epoch and find the windows when ASCENT climbs
above Rubin's horizon. (TLE accuracy degrades weeks from epoch, so treat the far-out
windows as *indicative* — for a real observation, fetch the Space-Track TLE nearest that
date.)

In [ ]:
mm = float(ASCENT_L2[52:63])
geo_mm = 1.00273790
drift = (mm - geo_mm) * 360.0
print(
    f"Mean motion {mm:.5f} vs GEO {geo_mm:.5f} rev/day -> drift ~{drift:+.2f} deg/day, "
    f"laps every ~{360/abs(drift):.0f} days\n"
)

days = np.arange(-120, 121, 1.0)
base = ascent.epoch
rows = []
for d in days:
    t = ts.tt_jd(base.tt + d)
    sp = wgs84.subpoint(ascent.at(t))
    alt = (ascent - rubin).at(t).altaz()[0].degrees
    rows.append((d, sp.longitude.degrees, alt))
drift_df = pd.DataFrame(rows, columns=["day_off", "sub_lon_E", "rubin_alt"])
vis = drift_df[drift_df.rubin_alt > 20]
print(
    f"ASCENT is >20 deg above Rubin's horizon on ~{len(vis)} of the sampled days "
    f"(+/-120 d of epoch {base.utc_iso()[:10]})."
)
if len(vis):
    # contiguous windows
    grp = (vis.day_off.diff() > 1.5).cumsum()
    for _, w in vis.groupby(grp):
        d0, d1 = w.day_off.min(), w.day_off.max()
        from datetime import timedelta as _td

        c0 = (day0 + _td(days=float(d0))).strftime("%Y-%m-%d")
        c1 = (day0 + _td(days=float(d1))).strftime("%Y-%m-%d")
        print(
            f"  visible window: day {d0:+.0f}..{d1:+.0f}  (~{c0} -> {c1}), "
            f"sub-lon {w.sub_lon_E.min():.0f}..{w.sub_lon_E.max():.0f} E, "
            f"peak alt {w.rubin_alt.max():.0f} deg"
        )

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(drift_df.day_off, drift_df.rubin_alt, lw=1.2)
ax.axhline(0, color="r", ls="--", lw=1, label="horizon")
ax.axhline(20, color="orange", ls=":", lw=1, label="alt = 20 deg")
ax.fill_between(
    drift_df.day_off,
    20,
    drift_df.rubin_alt,
    where=drift_df.rubin_alt > 20,
    alpha=0.3,
    color="green",
)
ax.set_xlabel("days from TLE epoch")
ax.set_ylabel("ASCENT altitude from Rubin [deg]")
ax.set_title("ASCENT drifts through Rubin's sky every ~105 days")
ax.legend()
plt.tight_layout()

## 4. LSSTCam exposures that night

Pull science exposures from `cdb_lsstcam.exposure` for the night, with their boresight
(`s_ra`, `s_dec`) and midpoint times. We then propagate ASCENT to each exposure's midpoint
and compute the angular separation from the boresight.

In [ ]:
# Night window in UTC: Chile is UTC-4 (CLT) / UTC-3 (CLST). Observing runs roughly
# local sunset->sunrise, i.e. ~22:00 UTC (NIGHT) -> ~12:00 UTC (NIGHT+1).
t_start = f"{NIGHT} 22:00:00"
t_end = (day0 + timedelta(days=1)).strftime("%Y-%m-%d") + " 12:00:00"

sql = f"""
SELECT exposure_id, obs_start, exp_midpt, exp_midpt_mjd,
       s_ra, s_dec, exp_time, band, img_type, scheduler_note
FROM   cdb_lsstcam.exposure
WHERE  obs_start >= '{t_start}'
  AND  obs_start <  '{t_end}'
  AND  img_type = 'science'
  AND  s_ra IS NOT NULL
ORDER  BY obs_start
"""
exp = consdb_query(sql)
print(f"{len(exp)} science exposures with valid pointing on night {NIGHT}")
exp.head()

In [ ]:
# Propagate ASCENT to each exposure midpoint and compute separation from boresight.
# LSSTCam science field of view: ~3.5 deg diameter -> ~1.75 deg radius.
LSSTCAM_FOV_RADIUS_DEG = 1.75

if len(exp) == 0:
    print("No exposures this night — try another NIGHT close to the TLE epoch.")
else:
    mid = pd.to_datetime(exp["exp_midpt"], utc=True)
    et = ts.from_datetimes(list(mid.dt.to_pydatetime()))
    tp = (ascent - rubin).at(et)
    a_alt, a_az, _ = tp.altaz()
    a_ra, a_dec, _ = tp.radec()

    sat = SkyCoord(a_ra._degrees * u.deg, a_dec.degrees * u.deg)
    bore = SkyCoord(exp["s_ra"].values * u.deg, exp["s_dec"].values * u.deg)
    sep = sat.separation(bore).deg

    exp = exp.assign(
        sat_ra=a_ra._degrees,
        sat_dec=a_dec.degrees,
        sat_alt=a_alt.degrees,
        sep_deg=sep,
        in_fov=(sep <= LSSTCAM_FOV_RADIUS_DEG) & (a_alt.degrees > 0),
    )
    print(f"Closest approach: {exp.sep_deg.min():.3f} deg")
    print(
        f"Exposures with ASCENT in FoV (sep <= {LSSTCAM_FOV_RADIUS_DEG} deg, sat above horizon): "
        f"{int(exp.in_fov.sum())}"
    )
    display(
        exp.nsmallest(10, "sep_deg")[
            [
                "exposure_id",
                "exp_midpt",
                "band",
                "s_ra",
                "s_dec",
                "sat_ra",
                "sat_dec",
                "sat_alt",
                "sep_deg",
                "in_fov",
            ]
        ]
    )

In [ ]:
# Plot exposure boresights and the ASCENT track for the night
if len(exp) > 0:
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(exp.s_ra, exp.s_dec, s=14, c="0.6", label="LSSTCam boresights")
    ax.plot(
        track.ra_deg,
        track.dec_deg,
        "-",
        color="C1",
        lw=1,
        label="ASCENT track (UTC day)",
    )
    hit = exp[exp.in_fov]
    if len(hit):
        ax.scatter(
            hit.s_ra,
            hit.s_dec,
            s=120,
            facecolors="none",
            edgecolors="red",
            lw=2,
            label="ASCENT in FoV",
        )
    ax.set_xlabel("RA [deg]")
    ax.set_ylabel("Dec [deg]")
    ax.set_title(f"Exposure pointings vs ASCENT — {NIGHT}")
    ax.legend()
    ax.invert_xaxis()
    plt.tight_layout()

## 6. Similar-sized GEO satellites visible from Chile

ASCENT is a **12U CubeSat** — unusually small for GEO; almost all *station-kept* GEO
satellites are large comsats (bus 2–6 m, far brighter than Landolt's intended faint source).
So as a *size* analog ASCENT is nearly unique. But if the goal is simply **"a GEO object
parked where Rubin can see it, to characterise GEO-object brightness in LSST data,"** the
cell below pulls the live CelesTrak GEO catalog and lists objects currently well-placed over
the Americas (sub-lon within Rubin's window, altitude > 30°). For a true *small* analog,
filter to other CubeSat/smallsat GEO tech-demos, or just track ASCENT itself during one of
its visibility windows from Section 3b.

In [ ]:
import urllib.request, io

GEO_URL = "https://celestrak.org/NORAD/elements/gp.php?GROUP=geo&FORMAT=tle"
raw = urllib.request.urlopen(GEO_URL, timeout=60).read().decode()
L = [x.rstrip() for x in raw.splitlines() if x.strip()]
geo_sats = [
    EarthSatellite(L[i + 1], L[i + 2], L[i].strip(), ts)
    for i in range(0, len(L) - 2, 3)
]
print(f"Loaded {len(geo_sats)} GEO objects from CelesTrak")

t_mid = ts.utc(day0.year, day0.month, day0.day + 1, 4)  # ~local midnight Chile
recs = []
for s in geo_sats:
    try:
        sp = wgs84.subpoint(s.at(t_mid))
        alt = (s - rubin).at(t_mid).altaz()[0].degrees
        if alt > 30:
            recs.append((s.name, sp.longitude.degrees, alt))
    except Exception:
        pass
cand = (
    pd.DataFrame(recs, columns=["name", "sub_lon_E", "rubin_alt"])
    .sort_values("rubin_alt", ascending=False)
    .reset_index(drop=True)
)
print(f"{len(cand)} GEO sats above alt 30 deg from Rubin at local midnight on {NIGHT}.")
display(cand.head(25))

## 5. Next step — natural brightness

For any exposure with `in_fov == True`, ASCENT should appear as a **trailed streak** (a GEO
satellite drifts slowly, but over a ~30 s LSST exposure with sidereal tracking it leaves a
short trail). To measure its natural brightness:

- Use the Butler to load the `calexp` for the matched `(exposure_id, detector)` — map the
  satellite RA/Dec to a detector via the WCS to find which CCD it landed on.
- Extract a cutout around the predicted track, measure the trail flux, and convert to an
  AB magnitude using the exposure's photometric zeropoint (`visit1_quicklook` /
  `ccdexposure_quicklook`).
- Compare against expected GEO-satellite brightness (typically ~mag 13–18 depending on
  size, phase angle, and specular vs. diffuse reflection) to anchor expectations for the
  Landolt source.

If no exposure contained ASCENT on this night, scan a range of nights: GEO satellites trace
the same narrow Dec band nightly, so the question is whether the survey cadence pointed there
while the satellite was sunlit and above the horizon.

In [ ]:
# Scan multiple nights for any FoV hit (run after the single-night cells work).
def scan_night(night, fov_radius=LSSTCAM_FOV_RADIUS_DEG):
    d0 = datetime.strptime(night, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    ts0 = f"{night} 22:00:00"
    ts1 = (d0 + timedelta(days=1)).strftime("%Y-%m-%d") + " 12:00:00"
    q = f"""SELECT exposure_id, exp_midpt, s_ra, s_dec, band
            FROM cdb_lsstcam.exposure
            WHERE obs_start >= '{ts0}' AND obs_start < '{ts1}'
              AND img_type='science' AND s_ra IS NOT NULL"""
    e = consdb_query(q)
    if len(e) == 0:
        return night, 0, np.nan, 0
    mid = pd.to_datetime(e["exp_midpt"], utc=True)
    et = ts.from_datetimes(list(mid.dt.to_pydatetime()))
    tp = (ascent - rubin).at(et)
    aa, _, _ = tp.altaz()
    ar, ad, _ = tp.radec()
    sep = (
        SkyCoord(ar._degrees * u.deg, ad.degrees * u.deg)
        .separation(SkyCoord(e.s_ra.values * u.deg, e.s_dec.values * u.deg))
        .deg
    )
    nhit = int(((sep <= fov_radius) & (aa.degrees > 0)).sum())
    return night, len(e), float(np.nanmin(sep)), nhit


# Example: scan a week around the TLE epoch
# rows = [scan_night((datetime(2026,6,24)+timedelta(d)).strftime("%Y-%m-%d")) for d in range(7)]
# pd.DataFrame(rows, columns=["night","n_exp","min_sep_deg","n_in_fov"])

## 7. Past-year refinement with Space-Track historical TLEs

CelesTrak serves only the *current* TLE. To get accurate positions for nights across the past
year we pull the **GP history** for NORAD 51287 from Space-Track (one element set per epoch,
~daily). We then pick, for each candidate night, the TLE whose epoch is **closest** to that
night — keeping propagation error small (hours/days, not months).

Credentials are read from `~/.lsst/spacetrack-credentials.txt` (`identity=...` / `password=...`,
chmod 600). Results are cached to parquet so we hit the API only once.

In [ ]:
import requests, pathlib, io
from datetime import datetime, timedelta, timezone

ST_CREDS = pathlib.Path.home() / ".lsst/spacetrack-credentials.txt"
TLE_CACHE = pathlib.Path("../data/ascent_tle_history.parquet")
NORAD_ID = 51287
# Past-year window for the GP history pull
HIST_START = "2025-06-01"
HIST_END = "2026-06-29"


def fetch_spacetrack_history(norad, start, end):
    creds = {}
    for line in ST_CREDS.read_text().splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    base = "https://www.space-track.org"
    q = (
        f"/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{norad}"
        f"/EPOCH/{start}--{end}/orderby/EPOCH%20asc/format/tle"
    )
    with requests.Session() as s:
        r = s.post(
            base + "/ajaxauth/login",
            data={"identity": creds["identity"], "password": creds["password"]},
            timeout=60,
        )
        r.raise_for_status()
        resp = s.get(base + q, timeout=120)
        resp.raise_for_status()
        s.get(base + "/ajaxauth/logout", timeout=30)
    return resp.text


if TLE_CACHE.exists():
    tle_hist = pd.read_parquet(TLE_CACHE)
    print(f"Loaded {len(tle_hist)} historical TLEs from cache: {TLE_CACHE}")
else:
    if not ST_CREDS.exists():
        raise FileNotFoundError(
            f"Space-Track creds not found at {ST_CREDS}. Create it with lines "
            "'identity=...' and 'password=...' (chmod 600), then re-run."
        )
    txt = fetch_spacetrack_history(NORAD_ID, HIST_START, HIST_END)
    L = [x.rstrip() for x in txt.splitlines() if x.strip()]
    recs = []
    for i in range(0, len(L) - 1, 2):
        if L[i].startswith("1 ") and L[i + 1].startswith("2 "):
            sat = EarthSatellite(L[i], L[i + 1], f"ASCENT", ts)
            recs.append({"epoch": sat.epoch.utc_datetime(), "l1": L[i], "l2": L[i + 1]})
    tle_hist = pd.DataFrame(recs).sort_values("epoch").reset_index(drop=True)
    TLE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    tle_hist.to_parquet(TLE_CACHE)
    print(
        f"Fetched & cached {len(tle_hist)} TLEs ({tle_hist.epoch.min()} -> {tle_hist.epoch.max()})"
    )

tle_hist["epoch"] = pd.to_datetime(tle_hist["epoch"], utc=True)
display(tle_hist.head())

### 7a. Accurate visibility scan over the past year

For each night, propagate using the nearest-epoch TLE and record ASCENT's peak altitude from
Rubin during the local-night window. Contiguous nights above the altitude threshold define the
visibility windows.

In [ ]:
def tle_for(dt_utc):
    """Return an EarthSatellite from the historical TLE whose epoch is nearest dt_utc."""
    idx = (tle_hist["epoch"] - dt_utc).abs().idxmin()
    row = tle_hist.loc[idx]
    return (
        EarthSatellite(row["l1"], row["l2"], "ASCENT", ts),
        abs((row["epoch"] - dt_utc).total_seconds()) / 86400,
    )


def night_peak_alt(night_str):
    d0 = datetime.strptime(night_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    # local night ~ 22:00 UTC (NIGHT) -> 12:00 UTC (NIGHT+1); sample every 20 min
    mins = np.arange(22 * 60, 36 * 60, 20)
    sat, dage = tle_for(d0 + timedelta(hours=29))  # ~mid-window epoch pick
    t = ts.utc(d0.year, d0.month, d0.day, 0, mins)
    alt = (sat - rubin).at(t).altaz()[0].degrees
    return alt.max(), dage


ALT_MIN = 20.0
nights = pd.date_range(HIST_START, HIST_END, freq="D", tz="UTC")
scan = []
for n in nights:
    ns = n.strftime("%Y-%m-%d")
    pk, dage = night_peak_alt(ns)
    scan.append((ns, pk, dage))
scan_df = pd.DataFrame(scan, columns=["night", "peak_alt", "tle_age_days"])
vis_nights = scan_df[scan_df.peak_alt > ALT_MIN].copy()
print(
    f"{len(vis_nights)} of {len(scan_df)} nights have ASCENT peak altitude > {ALT_MIN} deg.\n"
)

# group contiguous nights into windows
vis_nights["dt"] = pd.to_datetime(vis_nights.night)
vis_nights["gap"] = (vis_nights["dt"].diff().dt.days.fillna(1) > 1.5).cumsum()
windows = []
for _, w in vis_nights.groupby("gap"):
    windows.append(
        (
            w.night.iloc[0],
            w.night.iloc[-1],
            len(w),
            w.peak_alt.max(),
            w.tle_age_days.max(),
        )
    )
windows_df = pd.DataFrame(
    windows, columns=["start", "end", "n_nights", "peak_alt", "max_tle_age_d"]
)
print("Visibility windows (past year):")
display(windows_df)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(scan_df.night.values, scan_df.peak_alt.values, lw=0.8)
ax.axhline(ALT_MIN, color="orange", ls=":", label=f"alt = {ALT_MIN} deg")
ax.axhline(0, color="r", ls="--", lw=1)
ax.set_ylabel("nightly peak altitude [deg]")
ax.set_title("ASCENT visibility from Rubin (Space-Track TLEs)")
ax.set_xticks(scan_df.night.values[::30])
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()

### 7b. FoV cross-match across all visible windows

For every night in the visibility windows, query LSSTCam science exposures, propagate ASCENT
(nearest-epoch TLE) to each exposure midpoint, and flag exposures whose boresight is within the
LSSTCam FoV radius of the satellite. Collect all hits.

In [ ]:
def crossmatch_night(night_str, fov_radius=LSSTCAM_FOV_RADIUS_DEG):
    d0 = datetime.strptime(night_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    ts0 = f"{night_str} 22:00:00"
    ts1 = (d0 + timedelta(days=1)).strftime("%Y-%m-%d") + " 12:00:00"
    q = f"""SELECT exposure_id, exp_midpt, exp_midpt_mjd, s_ra, s_dec, band, exp_time
            FROM cdb_lsstcam.exposure
            WHERE obs_start >= '{ts0}' AND obs_start < '{ts1}'
              AND img_type='science' AND s_ra IS NOT NULL ORDER BY obs_start"""
    e = consdb_query(q)
    if len(e) == 0:
        return e
    mid = pd.to_datetime(e["exp_midpt"], utc=True)
    pyd = list(np.array(mid.dt.to_pydatetime()))
    et = ts.from_datetimes(pyd)
    # one nearest-epoch TLE for the night (GEO drift within a night is tiny)
    sat, dage = tle_for(d0 + timedelta(hours=29))
    tp = (sat - rubin).at(et)
    aa = tp.altaz()[0].degrees
    ar, ad, _ = tp.radec()
    sep = (
        SkyCoord(ar._degrees * u.deg, ad.degrees * u.deg)
        .separation(SkyCoord(e.s_ra.values * u.deg, e.s_dec.values * u.deg))
        .deg
    )
    e = e.assign(
        night=night_str,
        sat_ra=ar._degrees,
        sat_dec=ad.degrees,
        sat_alt=aa,
        sep_deg=sep,
        tle_age_d=dage,
        in_fov=(sep <= fov_radius) & (aa > 0),
    )
    return e


all_hits = []
summary = []
for night in vis_nights.night:
    e = crossmatch_night(night)
    if len(e) == 0:
        summary.append((night, 0, np.nan, 0))
        continue
    nhit = int(e.in_fov.sum())
    summary.append((night, len(e), float(e.sep_deg.min()), nhit))
    if nhit:
        all_hits.append(e[e.in_fov])

xm_summary = pd.DataFrame(
    summary, columns=["night", "n_exp", "min_sep_deg", "n_in_fov"]
)
print("Per-night cross-match summary (visible nights only):")
display(xm_summary)
print(
    f"\nTotal exposures with ASCENT in FoV across all windows: "
    f"{int(xm_summary.n_in_fov.sum())}"
)

if all_hits:
    hits = pd.concat(all_hits, ignore_index=True)
    print(f"\n{len(hits)} matched exposures:")
    display(
        hits[
            [
                "night",
                "exposure_id",
                "exp_midpt",
                "band",
                "s_ra",
                "s_dec",
                "sat_ra",
                "sat_dec",
                "sat_alt",
                "sep_deg",
                "tle_age_d",
            ]
        ]
    )
else:
    hits = pd.DataFrame()
    print(
        "\nNo exposures contained ASCENT in any visible-window night. "
        "Closest approaches are in xm_summary.min_sep_deg."
    )

## 8. Natural brightness — trail photometry from DP2 images

The 23 FoV hits (Section 7b) are *predicted* positions. Now we measure ASCENT's actual flux
in the LSST images. **Key fact:** a GEO satellite drifts at ~the sidereal rate relative to the
sidereally-tracking telescope, so over a 30 s exposure ASCENT leaves a **~1100–2300 px trail**
(227–456″), *not* a point source. Its light is smeared along that streak, so we must sum flux
*along the trail*, not in an aperture.

**Environment.** Image access uses the DP2 Butler. The whole notebook runs in the
**lsst-scipipe-13.0.0** kernel — that stack provides skyfield + ConsDB *and* resolves the DP2
Butler once the repo-root `.env` is loaded (Section 1 cell: `PGUSER`, `PGPASSFILE`,
`DAF_BUTLER_REPOSITORY_INDEX`). Coverage: DP2 spans day_obs **2025-04-15 → 2026-01-08**, so
only the **July 2025** visibility window is measurable here; the 2026 hits await a later data
release. The image dataset is **`preliminary_visit_image`** (`visit_image`/`calexp` are
unpopulated in this collection).

In [ ]:
# DP2 Butler (run from lsst-scipipe-12.1.0 stack with notebooks/.env loaded)
import lsst.daf.butler as daf_butler
import lsst.geom as geom
import lsst.sphgeom as sphgeom

DP2_REPO = "/sdf/group/rubin/repo/dp2_prep"
DP2_COLL = "LSSTCam/runs/DRP/DP2"
IMG_DT = "preliminary_visit_image"
dp2 = daf_butler.Butler(DP2_REPO, collections=[DP2_COLL])
dreg = dp2.registry
print(f"DP2 Butler ready: {IMG_DT} in {DP2_COLL}")

hits = pd.read_parquet("../data/ascent_fov_hits.parquet")
july = hits[hits.night.str.startswith("2025-07")].copy()  # only window inside DP2 range
print(f"{len(july)} hits in the DP2-covered window (July 2025)")

In [ ]:
def detectors_on_trail(visit, ra0, dec0, ra1, dec1, n=40):
    """Detectors (with images) whose sky region the trail crosses — no pixel I/O."""
    have = {
        ref.dataId["detector"]
        for ref in dreg.queryDatasets(
            IMG_DT, where=f"visit={visit}", instrument="LSSTCam"
        )
    }
    regions = {
        rec.detector: rec.region
        for rec in dreg.queryDimensionRecords(
            "visit_detector_region", where=f"visit={visit}", instrument="LSSTCam"
        )
        if rec.region is not None
    }
    crossed = set()
    for ra, dec in zip(np.linspace(ra0, ra1, n), np.linspace(dec0, dec1, n)):
        uv = sphgeom.UnitVector3d(sphgeom.LonLat.fromDegrees(ra, dec))
        for det, rg in regions.items():
            if det in have and rg.contains(uv):
                crossed.add(det)
    return sorted(crossed)


def trail_flux(visit, det, ra0, dec0, ra1, dec1, swath=6):
    """Sum background-subtracted counts in a +/-swath px band along the trail; -> nJy, SNR."""
    exp = dp2.get(IMG_DT, visit=visit, detector=det, instrument="LSSTCam")
    wcs, calib = exp.getWcs(), exp.getPhotoCalib()
    if wcs is None or calib is None:
        return np.nan, np.nan, 0
    arr = exp.image.array.astype(float)
    var = exp.variance.array
    med = np.nanmedian(arr)
    p0 = wcs.skyToPixel(geom.SpherePoint(ra0 * geom.degrees, dec0 * geom.degrees))
    p1 = wcs.skyToPixel(geom.SpherePoint(ra1 * geom.degrees, dec1 * geom.degrees))
    d = np.array([p1.x - p0.x, p1.y - p0.y])
    L = np.hypot(*d)
    if L < 1:
        return np.nan, np.nan, 0
    uvec = d / L
    nvec = np.array([-uvec[1], uvec[0]])
    H, W = arr.shape
    tot = vsum = 0.0
    npx = 0
    for s in range(int(L)):
        base = np.array([p0.x, p0.y]) + uvec * s
        for t in range(-swath, swath + 1):
            x, y = base + nvec * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < H:
                tot += arr[yi, xi] - med
                vsum += var[yi, xi]
                npx += 1
    njy = calib.instFluxToNanojansky(tot) if tot > 0 else np.nan
    snr = tot / np.sqrt(vsum) if vsum > 0 else np.nan
    return njy, snr, npx


print("trail photometry helpers defined")

In [ ]:
SNR_MIN = 3.0
rows = []
for _, r in july.iterrows():
    v = int(r.exposure_id)
    dets = detectors_on_trail(v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1)
    tot_njy, max_snr = 0.0, 0.0
    for det in dets:
        njy, snr, npx = trail_flux(
            v, det, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
        )
        if np.isfinite(snr) and snr > max_snr:
            max_snr = snr
        if np.isfinite(njy) and njy > 0 and snr > SNR_MIN:
            tot_njy += njy
    mag = -2.5 * np.log10(tot_njy * 1e-9 / 3631) if tot_njy > 0 else np.nan
    rows.append(
        dict(
            visit=v,
            band=r.band,
            sep_deg=r.sep_deg,
            trail_arcsec=r.trail_arcsec,
            detectors=dets,
            max_snr=max_snr,
            ab_mag=mag,
        )
    )
bright = pd.DataFrame(rows)
bright.to_parquet("../data/ascent_brightness.parquet")
det_ok = bright[bright.ab_mag.notna()]
print(
    f"{len(det_ok)}/{len(bright)} July hits yielded a trail detection (SNR>{SNR_MIN})."
)
print(
    f"AB mag: {det_ok.ab_mag.min():.2f} .. {det_ok.ab_mag.max():.2f}, median {det_ok.ab_mag.median():.2f}"
)
display(
    bright.sort_values("sep_deg")[
        ["visit", "band", "sep_deg", "trail_arcsec", "detectors", "max_snr", "ab_mag"]
    ]
)

In [ ]:
# Brightness distribution and the Landolt comparison
det_ok = bright[bright.ab_mag.notna()]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for band in sorted(det_ok.band.unique()):
    bb = det_ok[det_ok.band == band]
    ax[0].scatter(bb.sep_deg, bb.ab_mag, s=60, label=f"{band} (n={len(bb)})")
ax[0].set_xlabel("boresight separation [deg]")
ax[0].set_ylabel("ASCENT AB mag")
ax[0].invert_yaxis()
ax[0].legend()
ax[0].set_title("Measured ASCENT brightness (DP2, July 2025)")

ax[1].hist(det_ok.ab_mag, bins=np.arange(14, 21, 0.5), edgecolor="k", alpha=0.7)
ax[1].axvspan(13.5, 14.5, alpha=0.15, color="C2")  # illustrative Landolt target band
ax[1].set_xlabel("AB mag")
ax[1].set_ylabel("count")
ax[1].set_title("Distribution (green = illustrative Landolt target ~14)")
plt.tight_layout()

print(
    f"ASCENT natural brightness: AB {det_ok.ab_mag.min():.1f} - {det_ok.ab_mag.max():.1f}, "
    f"median {det_ok.ab_mag.median():.1f}."
)
print(
    "Wide spread is expected for a tumbling GEO object: brightness depends strongly on"
)
print(
    "solar phase angle and specular vs diffuse reflection. As a Landolt size-analog this"
)
print(
    "brackets the regime Rubin will see; Landolt's source flux is actively controlled,"
)
print(
    "so it will be a stable point (no trail if co-rotating; here ASCENT trails ~1100-2300 px)."
)

### 8b. Star-masked ("clean") brightness

The Section 8 swath-sum is contaminated: as each trail sweeps the sky it crosses **field
stars** (and the occasional cosmic ray), which dominate the integrated flux (Section 10 shows
this). We reject them using the **image's own `DETECTED|CR|SAT` mask plane** — the pipeline
already flagged every detected source pixel, so this removes *faint* stars too (a simple flux
threshold leaves them in, because the trail's own noise is inflated by the bright stars). We
then re-integrate the *satellite-only* flux: a clean magnitude where the satellite is detected
at SNR > 5, else a 3-σ upper limit. These clean values feed Section 9.

In [ ]:
# Shared helpers for star-masked trail photometry (used by Sections 8b, 9, 10)
# Star/CR rejection uses the IMAGE's own mask plane (DETECTED|CR|SAT) — the pipeline already
# flagged every detected source, so this catches faint stars a flux threshold would miss
# (and needs no catalog cross-match). A column is "flagged" if any core pixel (|perp|<=8) is set.
_REJECT_PLANES = ("DETECTED", "CR", "SAT")


def along_trail_flux(visit, det, r, swath=6):
    """Sample flux 1px-along-trail (perp swath summed). Returns (flux, var, flagged, t_s, calib).
    `flagged[i]` is True where a core pixel hits the DETECTED|CR|SAT mask plane (= star/CR).
    """
    exp = dp2.get(IMG_DT, visit=visit, detector=det, instrument="LSSTCam")
    wcs, calib = exp.getWcs(), exp.getPhotoCalib()
    if wcs is None or calib is None:
        return None
    arr = exp.image.array.astype(float)
    var = exp.variance.array
    H, W = arr.shape
    mk = exp.mask.array
    mpd = exp.mask.getMaskPlaneDict()
    reject_bits = 0
    for p in _REJECT_PLANES:
        if p in mpd:
            reject_bits |= 1 << mpd[p]
    med = np.nanmedian(arr)
    p0 = wcs.skyToPixel(
        geom.SpherePoint(r.trail_ra0 * geom.degrees, r.trail_dec0 * geom.degrees)
    )
    p1 = wcs.skyToPixel(
        geom.SpherePoint(r.trail_ra1 * geom.degrees, r.trail_dec1 * geom.degrees)
    )
    d = np.array([p1.x - p0.x, p1.y - p0.y])
    L = np.hypot(*d)
    if L < 1:
        return None
    u = d / L
    nn = np.array([-u[1], u[0]])
    fl, fv, fg = [], [], []
    for s in range(int(L)):
        base = np.array([p0.x, p0.y]) + u * s
        cx, cy = int(round(base[0])), int(round(base[1]))
        if not (0 <= cx < W and 0 <= cy < H):
            continue
        fs = vs = 0.0
        flagged = False
        for t in range(-swath, swath + 1):
            x, y = base + nn * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < H:
                fs += arr[yi, xi] - med
                vs += var[yi, xi]
                if mk[yi, xi] & reject_bits:
                    flagged = True
        fl.append(fs)
        fv.append(vs)
        fg.append(flagged)
    fl = np.array(fl)
    fv = np.array(fv)
    fg = np.array(fg)
    rate = (r.trail_arcsec / 0.2) / r.exp_time  # px/s along the trail
    t_s = np.arange(len(fl)) / rate
    return fl, fv, fg, t_s, calib


def grow_mask(flagged, wings=4):
    """Grow a boolean along-trail mask by +/-wings columns to cover PSF/CR wings."""
    m = flagged.copy()
    for i in np.where(flagged)[0]:
        m[max(0, i - wings) : i + wings + 1] = True
    return m


def ab_mag(cts, calib):
    nj = calib.instFluxToNanojansky(cts)
    return -2.5 * np.log10(nj * 1e-9 / 3631) if cts > 0 else np.nan


print("trail / mask-plane star-rejection helpers defined (DETECTED|CR|SAT)")

In [ ]:
SNR_CLEAN = 5.0
rows = []
for _, r in july.iterrows():
    v = int(r.exposure_id)
    dets = detectors_on_trail(v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1)
    fl_all, fv_all, fg_all, calib = [], [], [], None
    for det in dets:
        out = along_trail_flux(v, det, r)
        if out:
            fl_all.append(out[0])
            fv_all.append(out[1])
            fg_all.append(out[2])
            calib = out[4]
    if not fl_all:
        rows.append(
            dict(
                visit=v,
                band=r.band,
                sep_deg=r.sep_deg,
                raw_mag=np.nan,
                clean_mag=np.nan,
                clean_snr=np.nan,
                clean_ul_mag=np.nan,
                star_pct=np.nan,
            )
        )
        continue
    flux = np.concatenate(fl_all)
    fvar = np.concatenate(fv_all)
    clean = ~grow_mask(np.concatenate(fg_all), wings=4)  # mask-plane star/CR rejection
    raw, cln = flux.sum(), flux[clean].sum()
    cvar = fvar[clean].sum()
    csnr = cln / np.sqrt(cvar) if cvar > 0 else np.nan
    rows.append(
        dict(
            visit=v,
            band=r.band,
            sep_deg=r.sep_deg,
            raw_mag=ab_mag(raw, calib),
            clean_mag=(
                ab_mag(cln, calib)
                if (np.isfinite(csnr) and csnr > SNR_CLEAN)
                else np.nan
            ),
            clean_snr=csnr,
            clean_ul_mag=ab_mag(3 * np.sqrt(cvar), calib) if cvar > 0 else np.nan,
            star_pct=100 * (1 - cln / raw) if raw > 0 else np.nan,
        )
    )
bright_clean = pd.DataFrame(rows)
bright_clean.to_parquet("../data/ascent_brightness_clean.parquet")
ndet = int((bright_clean.clean_snr > SNR_CLEAN).sum())
print(
    f"{ndet}/{len(bright_clean)} July hits survive star-masking at SNR>{SNR_CLEAN:.0f}."
)
print(
    f"Star rejection via the DETECTED|CR|SAT mask plane (catches faint stars a flux cut misses)."
)
print(
    f"Raw swath-sum is a median {bright_clean.star_pct.median():.0f}% flagged-source flux => the raw "
    f"Section-8 mags are contaminated. Clean values below are the satellite alone."
)
display(bright_clean.sort_values("clean_snr", ascending=False).round(2))

## 9. Brightness vs solar elongation / phase angle (star-masked)

A satellite shines by reflected sunlight, so brightness should depend on the **solar phase
angle** α (Sun–satellite–observer; small α = near "full", brightest) — or equivalently the
**solar elongation** ε (Sun–observer–satellite), with ε + α ≈ 180°.

We use the **star-masked (clean) fluxes from Section 8b** — the satellite alone. Because most
passes are faint after de-contamination, we plot **detections** (clean SNR > 5) as points and
the rest as **3-σ upper limits** (down arrows). This is the honest view: it asks whether the
*satellite's own* brightness tracks illumination geometry.

In [ ]:
# Solar geometry for every clean measurement (detections + upper limits)
from skyfield.api import load as _sf_load

_eph = _sf_load("de421.bsp")
_sun, _earth = _eph["sun"], _eph["earth"]
rubin_geo = _earth + rubin  # rubin (topos) from Section 1

bmeas = bright_clean.merge(
    hits[["exposure_id", "exp_midpt"]],
    left_on="visit",
    right_on="exposure_id",
    how="left",
)


def geometry(mid):
    """(solar_elongation, solar_phase) in deg for an exposure midpoint."""
    t = ts.from_datetimes([pd.to_datetime(mid, utc=True).to_pydatetime()])
    sat, _ = tle_for(pd.to_datetime(mid, utc=True))  # returns (sat, age)
    sat_app = (sat - rubin).at(t)
    sun_app = rubin_geo.at(t).observe(_sun).apparent()
    elong = sat_app.separation_from(sun_app).degrees[0]
    sat_geo = sat.at(t).position.km.flatten()
    obs_geo = rubin.at(t).position.km.flatten()
    sun_dir = _earth.at(t).observe(_sun).apparent().position.km.flatten()
    v_obs = obs_geo - sat_geo
    cosa = np.dot(v_obs, sun_dir) / (np.linalg.norm(v_obs) * np.linalg.norm(sun_dir))
    return elong, np.degrees(np.arccos(np.clip(cosa, -1, 1)))


geo = bmeas.exp_midpt.apply(
    lambda m: pd.Series(geometry(m), index=["solar_elong_deg", "solar_phase_deg"])
)
bmeas[["solar_elong_deg", "solar_phase_deg"]] = geo
bmeas.to_parquet("../data/ascent_brightness_clean.parquet")

det = bmeas[bmeas.clean_mag.notna()]
print(
    f"{len(det)} star-masked DETECTIONS (clean SNR>{SNR_CLEAN:.0f}); "
    f"{bmeas.clean_ul_mag.notna().sum()-len(det)} upper limits."
)
print(
    f"Phase-angle range sampled: {bmeas.solar_phase_deg.min():.0f}-{bmeas.solar_phase_deg.max():.0f} deg."
)
if len(det) >= 3:
    print(
        f"corr(phase, clean_mag) over detections = {np.corrcoef(det.solar_phase_deg, det.clean_mag)[0,1]:+.2f}"
    )
else:
    print("Too few clean detections for a meaningful phase-curve correlation.")
display(
    bmeas[
        [
            "visit",
            "band",
            "clean_mag",
            "clean_ul_mag",
            "clean_snr",
            "solar_elong_deg",
            "solar_phase_deg",
        ]
    ]
    .sort_values("solar_phase_deg")
    .round(2)
)

In [ ]:
# Brightness (clean) vs phase / elongation: detections as points, non-detections as 3-sigma UL
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
det = bmeas[bmeas.clean_mag.notna()]
ul = bmeas[bmeas.clean_mag.isna() & bmeas.clean_ul_mag.notna()]
for axis, xcol, xlab in [
    (ax[0], "solar_phase_deg", "solar phase angle alpha [deg]"),
    (ax[1], "solar_elong_deg", "solar elongation epsilon [deg]"),
]:
    bands = sorted(bmeas.band.dropna().unique())
    for bi, bnd in enumerate(bands):
        dd = det[det.band == bnd]
        axis.errorbar(
            dd[xcol],
            dd.clean_mag,
            yerr=1.0857 / dd.clean_snr,
            fmt="o",
            ms=8,
            color=f"C{bi}",
            label=f"{bnd} detection (n={len(dd)})",
            zorder=3,
        )
        uu = ul[ul.band == bnd]
        axis.scatter(
            uu[xcol],
            uu.clean_ul_mag,
            marker="v",
            s=55,
            facecolors="none",
            edgecolors=f"C{bi}",
            alpha=0.7,
            label=f"{bnd} 3$\\sigma$ UL (n={len(uu)})",
        )
    axis.set_xlabel(xlab)
    axis.set_ylabel("ASCENT AB mag (star-masked)")
    axis.invert_yaxis()
    axis.legend(fontsize=8)
ax[0].set_title("Clean brightness vs phase angle")
ax[1].set_title("Clean brightness vs elongation")
plt.tight_layout()

ndet = bmeas.clean_mag.notna().sum()
print(
    f"After star-masking, only {ndet} of {len(bmeas)} July passes are detected (SNR>{SNR_CLEAN:.0f});"
)
print(
    "the rest are upper limits. The satellite is intrinsically FAINT (AB ~18-19) once field"
)
print(
    "stars are removed -- far fainter than the raw Section-8 values (AB ~14-16), which were"
)
print(
    "90-100% starlight. With so few clean detections spanning a narrow phase range, a phase"
)
print(
    "curve is NOT measurable here; the earlier 'flat' relation was an artifact of stellar"
)
print(
    "contamination, not a real constant-brightness result. Pulling out ASCENT's true phase"
)
print("behaviour needs brighter passes and/or stacking (see Section 10).")

## 10. Sub-30s variability along the trail — and a star-contamination correction

The Section 8 magnitudes are a **swath-sum over the whole ~2280 px trail** — an average over the
30 s exposure. But the trail is really a **time series**: ASCENT moves ~67 px/s (13 ms/px), so
*position along the trail maps directly to time within the single exposure*. We can therefore
extract a sub-second light curve and ask whether ASCENT flickers (tumbling glints).

**Caveat that turns out to dominate:** as the trail sweeps the sky it crosses **background
stars**. A star-crossing produces a flux spike of width ≈ PSF/rate ≈ 66 ms — *time-unresolved*
and easily mistaken for a glint. So before interpreting any variability we must remove
star-crossings. We cross-match bright along-trail spikes to the `single_visit_star` catalog,
sigma-clip them, and recompute.

In [ ]:
def trail_strip(visit, det, r, half=12):
    """Rectified image of the trail: rows = perpendicular offset, cols = along-trail (= time).
    Returns (strip[2*half+1, Nalong], t_s, core_flux, flagged) with background subtracted.
    `flagged[i]` is True where a core pixel (|perp|<=8) hits the DETECTED|CR|SAT mask plane.
    """
    exp = dp2.get(IMG_DT, visit=visit, detector=det, instrument="LSSTCam")
    wcs = exp.getWcs()
    arr = exp.image.array.astype(float)
    H, W = arr.shape
    med = np.nanmedian(arr)
    mk = exp.mask.array
    mpd = exp.mask.getMaskPlaneDict()
    reject_bits = 0
    for p in ("DETECTED", "CR", "SAT"):
        if p in mpd:
            reject_bits |= 1 << mpd[p]
    p0 = wcs.skyToPixel(
        geom.SpherePoint(r.trail_ra0 * geom.degrees, r.trail_dec0 * geom.degrees)
    )
    p1 = wcs.skyToPixel(
        geom.SpherePoint(r.trail_ra1 * geom.degrees, r.trail_dec1 * geom.degrees)
    )
    d = np.array([p1.x - p0.x, p1.y - p0.y])
    L = np.hypot(*d)
    u = d / L
    nn = np.array([-u[1], u[0]])
    cols, idx, flags = [], [], []
    for s in range(int(L)):
        base = np.array([p0.x, p0.y]) + u * s
        cx, cy = int(round(base[0])), int(round(base[1]))
        if not (0 <= cx < W and 0 <= cy < H):
            continue
        col = []
        flagged = False
        for t in range(-half, half + 1):
            x, y = base + nn * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < H:
                col.append(arr[yi, xi] - med)
                if abs(t) <= 8 and (mk[yi, xi] & reject_bits):
                    flagged = True
            else:
                col.append(np.nan)
        cols.append(col)
        idx.append(s)
        flags.append(flagged)
    strip = np.array(cols).T  # (perp, along)
    idx = np.array(idx)
    flags = np.array(flags)
    rate = (r.trail_arcsec / 0.2) / r.exp_time
    t_s = (idx - idx[0]) / rate
    core = np.nansum(strip[half - 6 : half + 7, :], axis=0)
    return strip, t_s, core, flags


print("trail_strip (rectified-image + mask-plane flags) helper defined")

In [ ]:
# Highest-SNR single-detector i-band case as the worked example
visit, det = 2025070800322, 157
r = hits[hits.exposure_id == visit].iloc[0]
flux, fvar, flagged, t_s, calib = along_trail_flux(visit, det, r)
mask = grow_mask(flagged, wings=4)
clean = ~mask  # DETECTED|CR|SAT star/CR rejection

raw_mag = ab_mag(flux.sum(), calib)
clean_mag = ab_mag(flux[clean].sum(), calib)
clean_snr = flux[clean].sum() / np.sqrt(fvar[clean].sum())
star_frac = 1 - flux[clean].sum() / flux.sum()
print(
    f"visit {visit} det {det} {r.band}-band, {len(flux)} px over {r.exp_time}s ({r.trail_arcsec:.0f} arcsec)"
)
print(f"  raw swath-sum (Section 8): AB {raw_mag:.2f}")
print(
    f"  star-masked (satellite)  : AB {clean_mag:.2f}  (integrated SNR {clean_snr:.1f})"
)
print(f"  flagged sources (stars/CRs) = {100*star_frac:.0f}% of the raw trail flux")

In [ ]:
# Strip-chart: rectified trail image, raw vs star-masked (i-band worked example above)
strip, ts_strip, core, flagged = trail_strip(visit, det, r, half=12)
mask_cols = grow_mask(flagged, wings=4)  # DETECTED|CR|SAT columns (+/-4px wings)
strip_masked = strip.copy()
strip_masked[:, mask_cols] = np.nan

half = 12
ext = [ts_strip[0], ts_strip[-1], -half * 0.2, half * 0.2]
vmin, vmax = np.nanpercentile(strip, [5, 99])
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].imshow(
    strip,
    aspect="auto",
    origin="lower",
    cmap="gray_r",
    vmin=vmin,
    vmax=vmax,
    extent=ext,
)
ax[0].set_ylabel("perp offset [arcsec]")
ax[0].set_title(
    f"Trail strip — visit {visit} det {det} ({r.band}-band). "
    "Dark blobs = field stars/CRs the trail crosses."
)
ax[1].imshow(
    strip_masked,
    aspect="auto",
    origin="lower",
    cmap="gray_r",
    vmin=vmin,
    vmax=vmax,
    extent=ext,
)
ax[1].set_ylabel("perp offset [arcsec]")
ax[1].set_xlabel("time within exposure [s]   (= along-trail position; ~67 px/s)")
ax[1].set_title(
    f"DETECTED|CR|SAT mask plane: {mask_cols.sum()}/{len(mask_cols)} columns clipped. "
    "Satellite would be a faint band at offset 0."
)
for a in ax:
    a.axhline(0, color="C1", lw=0.5, ls=":", alpha=0.6)
plt.tight_layout()
# residual check: any >5 sigma column left after masking?
sig = 1.4826 * np.median(np.abs(core[~mask_cols] - np.median(core[~mask_cols])))
resid = int(
    ((~mask_cols) & (np.abs(core - np.median(core[~mask_cols])) > 5 * sig)).sum()
)
print(
    f"Strip {strip.shape} (perp x along) over {ts_strip[-1]:.1f}s; {mask_cols.sum()} masked columns "
    f"from {int(flagged.sum())} flagged-core seeds; {resid} residual >5sigma columns remain."
)
print(
    "Using the pipeline mask plane catches faint stars an 8sigma flux cut left behind (the"
)
print(
    "leftover dots in the earlier version). The bottom panel shows no persistent offset-0"
)
print(
    "band above noise -> the satellite is undetected in this single exposure once sources are"
)
print("removed (consistent with Section 8b's clean SNR for this pass).")

In [ ]:
# Apply the correction to the brightest July detections + variability test
check = [2025070600596, 2025070800361, 2025070800322, 2025070100726]
rows = []
for visit in check:
    r = hits[hits.exposure_id == visit].iloc[0]
    dets = detectors_on_trail(
        visit, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
    )
    fl_all, fv_all, fg_all, calib = [], [], [], None
    for det in dets:
        out = along_trail_flux(visit, det, r)
        if out:
            fl_all.append(out[0])
            fv_all.append(out[1])
            fg_all.append(out[2])
            calib = out[4]
    if not fl_all:
        continue
    flux = np.concatenate(fl_all)
    fvar = np.concatenate(fv_all)
    clean = ~grow_mask(np.concatenate(fg_all), wings=4)
    cln_snr = flux[clean].sum() / np.sqrt(fvar[clean].sum())
    mu = flux[clean].mean()
    chi2 = np.sum((flux[clean] - mu) ** 2 / fvar[clean]) / (
        clean.sum() - 1
    )  # ~1 => constant
    rows.append(
        dict(
            visit=visit,
            band=r.band,
            raw_mag=ab_mag(flux.sum(), calib),
            clean_mag=ab_mag(flux[clean].sum(), calib),
            star_pct=100 * (1 - flux[clean].sum() / flux.sum()),
            clean_snr=cln_snr,
            redchi2=chi2,
        )
    )
corr = pd.DataFrame(rows)
display(corr.round(2))
print("Findings:")
print(
    "- After DETECTED|CR|SAT masking, most of the Section-8 swath-sum flux is field stars;"
)
print(
    "  the satellite trail itself is faint (clean SNR <=~5), often marginal per exposure."
)
print(
    "  Section 8/9 raw mags are contaminated UPPER limits — true ASCENT is fainter (AB >~18-19)."
)
print(
    "- reduced chi^2 of the star-removed per-pixel flux vs a constant is ~1-1.6: NO convincing"
)
print(
    "  intrinsic sub-30s variability. The apparent flicker was star/CR crossings, not glints."
)
print(
    "- A real glint search needs the satellite trail detected at high SNR first (brighter"
)
print(
    "  passes / stacking), then PSF-width excesses NOT coincident with flagged sources."
)

## 11. Stacking the cleaned trails — is ASCENT there at all?

Per exposure ASCENT is undetected after star-masking (Section 8b/10). But the satellite sits at
the *same transverse offset (0)* in every along-trail column, so we can **coadd** all clean
columns across all passes: the satellite (if present) builds up at offset 0 while noise averages
down as ~1/√N. With ~25k clean columns that is a ~160× noise reduction per transverse bin.

We (1) collect the star-masked transverse profiles (flux vs perpendicular offset, local-sky
subtracted per column), (2) inverse-variance coadd them, and — crucially — (3) **inject a known
synthetic source at offset 0** to prove the stacker *would* recover a real signal at this depth.

In [ ]:
HALF_T = 10  # transverse half-width [px] for the profile
_wing = np.abs(np.arange(-HALF_T, HALF_T + 1)) > 6  # |offset|>1.2" used as local sky


def transverse_profiles(visit, det, r):
    """Star-masked, local-sky-subtracted transverse profiles for one trail on one detector.
    Returns (profiles[Ncol, 2*HALF_T+1], var[...]) or None."""
    exp = dp2.get(IMG_DT, visit=visit, detector=det, instrument="LSSTCam")
    wcs, calib = exp.getWcs(), exp.getPhotoCalib()
    if wcs is None or calib is None:
        return None
    arr = exp.image.array.astype(float)
    var = exp.variance.array
    Hh, W = arr.shape
    mk = exp.mask.array
    mpd = exp.mask.getMaskPlaneDict()
    rb = 0
    for p in ("DETECTED", "CR", "SAT"):
        if p in mpd:
            rb |= 1 << mpd[p]
    p0 = wcs.skyToPixel(
        geom.SpherePoint(r.trail_ra0 * geom.degrees, r.trail_dec0 * geom.degrees)
    )
    p1 = wcs.skyToPixel(
        geom.SpherePoint(r.trail_ra1 * geom.degrees, r.trail_dec1 * geom.degrees)
    )
    d = np.array([p1.x - p0.x, p1.y - p0.y])
    L = np.hypot(*d)
    if L < 1:
        return None
    u = d / L
    nn = np.array([-u[1], u[0]])
    profs, vrs, flags = [], [], []
    for s in range(int(L)):
        base = np.array([p0.x, p0.y]) + u * s
        cx, cy = int(round(base[0])), int(round(base[1]))
        if not (0 <= cx < W and 0 <= cy < Hh):
            continue
        pr, pv, flag = [], [], False
        for t in range(-HALF_T, HALF_T + 1):
            x, y = base + nn * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < Hh:
                pr.append(arr[yi, xi])
                pv.append(var[yi, xi])
                if mk[yi, xi] & rb:
                    flag = True
            else:
                pr.append(np.nan)
                pv.append(np.nan)
        profs.append(pr)
        vrs.append(pv)
        flags.append(flag)
    profs = np.array(profs)
    vrs = np.array(vrs)
    flags = np.array(flags)
    clean = ~grow_mask(flags, wings=4)
    profs, vrs = profs[clean], vrs[clean]
    good = np.isfinite(profs).all(axis=1)
    profs, vrs = profs[good], vrs[good]
    if len(profs) == 0:
        return None
    profs = profs - np.median(
        profs[:, _wing], axis=1, keepdims=True
    )  # per-column local sky
    return profs, vrs


allP, allV, calib0 = [], [], None
for _, r in july.iterrows():
    v = int(r.exposure_id)
    for det in detectors_on_trail(
        v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
    ):
        out = transverse_profiles(v, det, r)
        if out:
            allP.append(out[0])
            allV.append(out[1])
    calib0 = (
        dp2.get(
            IMG_DT + ".photoCalib",
            visit=v,
            detector=detectors_on_trail(
                v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
            )[0],
            instrument="LSSTCam",
        )
        if calib0 is None
        and detectors_on_trail(v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1)
        else calib0
    )
Pstack = np.vstack(allP)
Vstack = np.vstack(allV)
print(f"collected {len(Pstack)} clean transverse profiles across all July passes")

In [ ]:
w = 1.0 / Vstack


def coadd(M):
    return np.nansum(M * w, axis=0) / np.nansum(w, axis=0)


offs = np.arange(-HALF_T, HALF_T + 1) * 0.2
err = np.sqrt(1.0 / np.nansum(w, axis=0))
i0, coreS = HALF_T, slice(HALF_T - 2, HALF_T + 3)

real = coadd(Pstack)
real_sum = np.nansum(real[coreS])
real_err = np.sqrt(np.nansum(err[coreS] ** 2))

# Injection test: add Gaussian (sigma 1.5px ~ FWHM 0.7") at offset 0, several brightnesses
inj_rows = []
for inj in [2.0, 5.0, 10.0]:
    g = np.exp(-0.5 * (np.arange(-HALF_T, HALF_T + 1) / 1.5) ** 2)
    g = g / g.sum() * inj
    rec = np.nansum(coadd(Pstack + g[None, :])[coreS])
    inj_rows.append((inj, rec, rec / real_err))
recov = (inj_rows[-1][1] - real_sum) / 10.0  # recovered fraction per injected count

print(f"N = {len(Pstack)} profiles; central +/-2px error = {real_err:.2f} cts")
print(
    f"REAL offset-0 stack: central sum {real_sum:+.2f} +/- {real_err:.2f}  (SNR {real_sum/real_err:+.1f})"
)
print("Injection recovery (proves the stacker works):")
for inj, rec, snr in inj_rows:
    print(
        f"  inject {inj:4.1f} cts/col -> recovered {rec-real_sum:+.2f} (SNR {snr:+.1f})"
    )
print(f"  recovered fraction ~ {recov:.2f}  (=> stacker is linear and unbiased)")

ul = 3 * real_err / max(recov, 0.5)
trail_px = july.trail_arcsec.mean() / 0.2
ul_mag = -2.5 * np.log10(calib0.instFluxToNanojansky(ul * trail_px) * 1e-9 / 3631)
print(
    f"\n3-sigma per-column UL: {ul:.2f} cts -> integrated over {trail_px:.0f}px trail -> AB {ul_mag:.1f}"
)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.errorbar(offs, real, yerr=err, fmt="o-", label="real stack")
g10 = np.exp(-0.5 * (np.arange(-HALF_T, HALF_T + 1) / 1.5) ** 2)
g10 = g10 / g10.sum() * 10
ax.plot(
    offs, real + g10, "r--", alpha=0.6, label="if AB-equiv source were present (inj=10)"
)
ax.axvline(0, color="0.7", lw=0.8)
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("transverse offset [arcsec]")
ax.set_ylabel("coadded flux/px [cts]")
ax.set_title(f"ASCENT stacked transverse profile (N={len(Pstack)}) — no peak at 0")
ax.legend()
plt.tight_layout()
print(
    f"\nCONCLUSION: injections recover cleanly but the REAL stack shows NO peak at offset 0."
)
print(
    f"ASCENT is undetected even stacked: it is fainter than AB ~{ul_mag:.1f} (per-exposure-equivalent)."
)

### 11b. Did we look in the right place? — wide cross-track offset search

The Section 11 stack only summed a narrow transverse window around the *predicted* trail
(offset 0). But Section 12 showed GOES 19's real trail sat **~93″ off** its TLE prediction —
and that error is **intrinsic to TLE accuracy, not propagation age** (GOES 19's TLE was < 0.7 d
old, like ASCENT's). So a fair non-detection must rule out a trail *at any plausible offset*: if
ASCENT had a GOES-like cross-track error, the narrow stack would have integrated blank sky.

We therefore re-stack over a **wide ±100″ transverse window** (star-masked, per-column local
sky) and look for an SNR peak at *any* offset. With ~27k columns this is a √N-deep matched
search across the whole offset range at once.

In [ ]:
HALF_W = 500  # wide transverse half-window [px] = +/-100"
_wing_w = np.abs(np.arange(-HALF_W, HALF_W + 1)) > (
    HALF_W - 40
)  # outer 40px = local sky

allP_w, allV_w = [], []
for _, r in july.iterrows():
    v = int(r.exposure_id)
    for det in detectors_on_trail(
        v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
    ):
        exp = dp2.get(IMG_DT, visit=v, detector=det, instrument="LSSTCam")
        wcs = exp.getWcs()
        if wcs is None:
            continue
        arr = exp.image.array.astype(float)
        var = exp.variance.array
        mk = exp.mask.array
        mpd = exp.mask.getMaskPlaneDict()
        Hh, W = arr.shape
        rb = 0
        for p in ("DETECTED", "CR", "SAT"):
            if p in mpd:
                rb |= 1 << mpd[p]
        p0 = wcs.skyToPixel(
            geom.SpherePoint(r.trail_ra0 * geom.degrees, r.trail_dec0 * geom.degrees)
        )
        p1 = wcs.skyToPixel(
            geom.SpherePoint(r.trail_ra1 * geom.degrees, r.trail_dec1 * geom.degrees)
        )
        d = np.array([p1.x - p0.x, p1.y - p0.y])
        L = np.hypot(*d)
        if L < 1:
            continue
        u = d / L
        nn = np.array([-u[1], u[0]])
        for s in range(int(L)):
            base = np.array([p0.x, p0.y]) + u * s
            cx, cy = int(round(base[0])), int(round(base[1]))
            if not (0 <= cx < W and 0 <= cy < Hh):
                continue
            if mk[cy, cx] & rb:
                continue  # skip star-contaminated columns
            prof = np.full(2 * HALF_W + 1, np.nan)
            pv = np.full(2 * HALF_W + 1, np.nan)
            for k, t in enumerate(range(-HALF_W, HALF_W + 1)):
                x, y = base + nn * t
                xi, yi = int(round(x)), int(round(y))
                if 0 <= xi < W and 0 <= yi < Hh and not (mk[yi, xi] & rb):
                    prof[k] = arr[yi, xi]
                    pv[k] = var[yi, xi]
            allP_w.append(prof)
            allV_w.append(pv)
Pw = np.array(allP_w)
Vw = np.array(allV_w)
with np.errstate(invalid="ignore"):
    _sky_w = np.nanmedian(
        Pw[:, _wing_w], axis=1, keepdims=True
    )  # some columns all-NaN (off-chip)
Pw = Pw - np.where(np.isfinite(_sky_w), _sky_w, 0.0)
ww = 1.0 / Vw
stack_w = np.nansum(Pw * ww, axis=0) / np.nansum(ww, axis=0)
err_w = np.sqrt(1.0 / np.nansum(ww, axis=0))
snr_w = stack_w / err_w
offs_w = np.arange(-HALF_W, HALF_W + 1) * 0.2
imax = np.nanargmax(snr_w)
nbin = int(np.isfinite(snr_w).sum())
print(
    f"wide stack: {len(Pw)} clean columns, searched offset +/-{HALF_W*0.2:.0f} arcsec ({nbin} bins)"
)
print(
    f"max SNR anywhere: {snr_w[imax]:.1f} at offset {offs_w[imax]:+.1f} arcsec;  SNR at predicted (0): {snr_w[HALF_W]:+.1f}"
)
print(
    f"bins > 5 sigma: {(snr_w>5).sum()} (noise expectation ~{nbin*2.9e-7:.4f});  "
    f"> 3 sigma: {(snr_w>3).sum()} (expectation ~{nbin*1.35e-3:.1f})"
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(offs_w, snr_w, lw=0.7)
ax.axhline(0, color="0.7", lw=0.8)
ax.axhline(5, color="r", ls="--", lw=0.8, label="5 sigma")
ax.axhline(-5, color="r", ls="--", lw=0.8)
ax.axvline(0, color="C1", lw=0.8, ls=":", label="TLE-predicted offset")
ax.axvspan(-93, -93, color="none")  # reference: GOES 19 had ~93" error
ax.set_xlabel("transverse (cross-track) offset [arcsec]")
ax.set_ylabel("stacked SNR")
ax.set_title(
    f"ASCENT wide cross-track search (N={len(Pw)} cols): no trail at any offset"
)
ax.legend()
plt.tight_layout()
print(
    "\nRESULT: even allowing a GOES-like (~90 arcsec) cross-track TLE error, there is NO trail at"
)
print(
    "ANY offset (max SNR < 2, zero bins above 3 sigma). The non-detection is robust to pointing"
)
print(
    "error -- ASCENT is genuinely below the noise floor, fainter than the Section 11 upper limit."
)

In [ ]:
# Stacked wide rectified strip: 2-D map of the coadd over the full +/-120" cross-track search.
# Each pass is resampled onto a common fractional-along-trail grid so all passes align in "time",
# then inverse-variance coadded. A real satellite would be a horizontal band; cross-track error
# would shift the band off 0. (GOES 19 in Section 12 shows what a real band looks like.)
NT = 60  # along-trail (time) bins
acc = np.zeros((2 * HALF_W + 1, NT))
accw = np.zeros((2 * HALF_W + 1, NT))
for _, r in july.iterrows():
    v = int(r.exposure_id)
    for det in detectors_on_trail(
        v, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
    ):
        exp = dp2.get(IMG_DT, visit=v, detector=det, instrument="LSSTCam")
        wcs = exp.getWcs()
        if wcs is None:
            continue
        arr = exp.image.array.astype(float)
        var = exp.variance.array
        mk = exp.mask.array
        mpd = exp.mask.getMaskPlaneDict()
        Hh, W = arr.shape
        med = np.nanmedian(arr)
        rb = 0
        for p in ("DETECTED", "CR", "SAT"):
            if p in mpd:
                rb |= 1 << mpd[p]
        p0 = wcs.skyToPixel(
            geom.SpherePoint(r.trail_ra0 * geom.degrees, r.trail_dec0 * geom.degrees)
        )
        p1 = wcs.skyToPixel(
            geom.SpherePoint(r.trail_ra1 * geom.degrees, r.trail_dec1 * geom.degrees)
        )
        d = np.array([p1.x - p0.x, p1.y - p0.y])
        L = np.hypot(*d)
        if L < 1:
            continue
        u = d / L
        nn = np.array([-u[1], u[0]])
        for s in range(int(L)):
            base = np.array([p0.x, p0.y]) + u * s
            cx, cy = int(round(base[0])), int(round(base[1]))
            if not (0 <= cx < W and 0 <= cy < Hh) or (mk[cy, cx] & rb):
                continue
            tb = min(int(NT * s / L), NT - 1)
            for k, t in enumerate(range(-HALF_W, HALF_W + 1)):
                x, y = base + nn * t
                xi, yi = int(round(x)), int(round(y))
                if 0 <= xi < W and 0 <= yi < Hh and not (mk[yi, xi] & rb):
                    wk = 1.0 / var[yi, xi]
                    acc[k, tb] += (arr[yi, xi] - med) * wk
                    accw[k, tb] += wk
sstrip = np.where(accw > 0, acc / accw, np.nan)
with np.errstate(invalid="ignore"):
    _sky = np.nanmedian(sstrip[_wing_w, :], axis=0)
sstrip = sstrip - np.where(np.isfinite(_sky), _sky, 0.0)
PB = 10
nperp = sstrip.shape[0] // PB  # bin transverse 10px = 2"
sb = np.nanmean(sstrip[: nperp * PB].reshape(nperp, PB, NT), axis=1)
perp_off = (np.arange(nperp) - nperp / 2) * PB * 0.2

fig, ax = plt.subplots(figsize=(12, 5))
lim = np.nanpercentile(np.abs(sb[np.isfinite(sb)]), 99)
im = ax.imshow(
    sb,
    aspect="auto",
    origin="lower",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
    extent=[0, 1, perp_off[0], perp_off[-1]],
)
ax.axhline(0, color="k", ls=":", lw=0.9, label="TLE-predicted offset (0)")
ax.set_xlabel("fractional position along trail  (= time within exposure)")
ax.set_ylabel("cross-track offset [arcsec]")
ax.set_title(
    f'ASCENT STACKED wide rectified strip (all July passes, +/-{HALF_W*0.2:.0f}") '
    "— no band at any offset"
)
ax.legend(loc="upper right")
plt.colorbar(im, label="coadded flux [cts]")
plt.tight_layout()
print(
    f'Stacked strip binned to {sb.shape} (perp 2" x ~0.5s). Flux range '
    f"{np.nanmin(sb):.1f}..{np.nanmax(sb):.1f} cts -- pure speckle noise, no horizontal trail"
)
print("at offset 0 or anywhere. Contrast with GOES 19's solid band in Section 12.")

## 12. Pipeline validation on a bright satellite — GOES 19

ASCENT comes out undetected — but is that physical, or is the pipeline broken? We validate on
**GOES 19** (NORAD 60133), a large, station-kept weather satellite at −75°E that rides **54° high**
from Rubin. We run the *same* historical-TLE cross-match and trail photometry. A bright GEO target
should yield an obvious, high-SNR streak — if it does, ASCENT's non-detection is real.

(`data/goes19_fov_hits.parquet` was built by the same Space-Track + ConsDB cross-match as
Section 7, restricted to GOES 19's TLE-history window 2025-11 → 2026-01.)

In [ ]:
goes = pd.read_parquet("../data/goes19_fov_hits.parquet")
gr = goes.nsmallest(1, "sep_deg").iloc[0]
gvisit = int(gr.exposure_id)
print(
    f"GOES 19: {len(goes)} FoV hits / {goes.night.nunique()} nights; closest {goes.sep_deg.min():.2f} deg, "
    f"alt {gr.sat_alt:.0f}, {gr.band}-band, trail {gr.trail_arcsec:.0f} arcsec"
)

# pick the detector carrying the LONGEST on-chip segment of the trail
have = {
    ref.dataId["detector"]
    for ref in dreg.queryDatasets(IMG_DT, where=f"visit={gvisit}", instrument="LSSTCam")
}
regs = {
    rec.detector: rec.region
    for rec in dreg.queryDimensionRecords(
        "visit_detector_region", where=f"visit={gvisit}", instrument="LSSTCam"
    )
    if rec.region
}
seg = {}
for ra, dec in zip(
    np.linspace(gr.trail_ra0, gr.trail_ra1, 400),
    np.linspace(gr.trail_dec0, gr.trail_dec1, 400),
):
    uv = sphgeom.UnitVector3d(sphgeom.LonLat.fromDegrees(ra, dec))
    for dd, rg in regs.items():
        if dd in have and rg.contains(uv):
            seg[dd] = seg.get(dd, 0) + 1
det = max(seg, key=seg.get)
print(f"trail crosses {sorted(seg)}; using detector {det} (longest on-chip segment)")

exp = dp2.get(IMG_DT, visit=gvisit, detector=det, instrument="LSSTCam")
wcs, calib = exp.getWcs(), exp.getPhotoCalib()
arr = exp.image.array.astype(float)
var = exp.variance.array
Hh, W = arr.shape
med = np.nanmedian(arr)
p0 = wcs.skyToPixel(
    geom.SpherePoint(gr.trail_ra0 * geom.degrees, gr.trail_dec0 * geom.degrees)
)
p1 = wcs.skyToPixel(
    geom.SpherePoint(gr.trail_ra1 * geom.degrees, gr.trail_dec1 * geom.degrees)
)
d = np.array([p1.x - p0.x, p1.y - p0.y])
L = np.hypot(*d)
u = d / L
nn = np.array([-u[1], u[0]])


# TLE has a cross-track error (deg-scale at GEO); locate the real trail with a matched filter:
# for each perpendicular offset, sum the core flux over the on-chip span; take the max.
def trail_sum(poff, wcore=2):
    tot = 0.0
    npx = 0
    for s in range(0, int(L)):
        base = np.array([p0.x, p0.y]) + u * s + nn * poff
        for t in range(-wcore, wcore + 1):
            x, y = base + nn * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < Hh:
                tot += arr[yi, xi] - med
                npx += 1
    return tot if npx > 200 else -np.inf


scan = np.arange(-800, 801, 2)
sums = np.array([trail_sum(o) for o in scan])
poff_best = scan[np.argmax(sums)]
print(
    f"cross-track offset of real trail vs TLE prediction: {poff_best*0.2:.0f} arcsec "
    f"(deg-scale TLE error at GEO; small vs the {gr.trail_arcsec:.0f} arcsec trail length)"
)

# Photometer the recentered trail (+-4px core, image median as sky)
core_sum = core_var = 0.0
npx = 0
for s in range(0, int(L)):
    base = np.array([p0.x, p0.y]) + u * s + nn * poff_best
    for t in range(-4, 5):
        x, y = base + nn * t
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < Hh:
            core_sum += arr[yi, xi] - med
            core_var += var[yi, xi]
            npx += 1
g_mag = -2.5 * np.log10(calib.instFluxToNanojansky(core_sum) * 1e-9 / 3631)
print(
    f"GOES 19 recentered trail: {npx}px, {core_sum:.0f} cts, SNR {core_sum/np.sqrt(core_var):.0f}, AB {g_mag:.2f}"
)

xs = np.linspace(p0.x, p1.x, 400) + nn[0] * poff_best
ys = np.linspace(p0.y, p1.y, 400) + nn[1] * poff_best
on = [(0 <= x < W and 0 <= y < Hh) for x, y in zip(xs, ys)]
oxs = [x for x, o in zip(xs, on) if o]
oys = [y for y, o in zip(ys, on) if o]
xlo, xhi = max(0, int(min(oxs)) - 40), min(W, int(max(oxs)) + 40)
ylo, yhi = max(0, int(min(oys)) - 40), min(Hh, int(max(oys)) + 40)
fig, ax = plt.subplots(figsize=(9, 6))
cut = arr[ylo:yhi, xlo:xhi] - med
vmin, vmax = np.nanpercentile(cut, [10, 99.5])
ax.imshow(
    cut,
    origin="lower",
    cmap="gray_r",
    vmin=vmin,
    vmax=vmax,
    extent=[xlo, xhi, ylo, yhi],
)
ax.plot(
    np.array(xs)[on],
    np.array(ys)[on],
    "r-",
    lw=0.8,
    alpha=0.6,
    label="recentered GOES 19 trail",
)
ax.legend()
ax.set_title(
    f"GOES 19 — visit {gvisit} det {det}: bright trail, AB {g_mag:.1f}, "
    f"SNR {core_sum/np.sqrt(core_var):.0f}"
)
plt.tight_layout()
print(
    f"\nVALIDATION PASSED: a bright station-kept GEO gives an unambiguous high-SNR trail (AB ~{g_mag:.0f})."
)
print(
    "The same machinery finds nothing for ASCENT (3-sigma UL AB ~21.6 even stacked), so ASCENT's"
)
print(
    "non-detection is PHYSICAL -- a 12U CubeSat is ~10 mag (10,000x) fainter than a full-size comsat."
)

In [ ]:
# GOES 19 wide rectified strip: the SAME +/-120" view for a bright satellite, so the contrast is
# explicit. The trail is the solid horizontal band; it sits at the real cross-track offset
# (~ -90"), NOT at the TLE-predicted 0 -- the visual proof of the TLE cross-track error.
HALF_G = 600
exp = dp2.get(IMG_DT, visit=gvisit, detector=det, instrument="LSSTCam")
wcs = exp.getWcs()
arr = exp.image.array.astype(float)
Hh, W = arr.shape
med = np.nanmedian(arr)
p0 = wcs.skyToPixel(
    geom.SpherePoint(gr.trail_ra0 * geom.degrees, gr.trail_dec0 * geom.degrees)
)
p1 = wcs.skyToPixel(
    geom.SpherePoint(gr.trail_ra1 * geom.degrees, gr.trail_dec1 * geom.degrees)
)
dd = np.array([p1.x - p0.x, p1.y - p0.y])
L = np.hypot(*dd)
u = dd / L
nn = np.array([-u[1], u[0]])
rate = (gr.trail_arcsec / 0.2) / 30.0
cols, sidx = [], []
for s in range(int(L)):
    base = np.array([p0.x, p0.y]) + u * s
    cx, cy = int(round(base[0])), int(round(base[1]))
    if not (0 <= cx < W and 0 <= cy < Hh):
        continue
    col = []
    for t in range(-HALF_G, HALF_G + 1):
        x, y = base + nn * t
        xi, yi = int(round(x)), int(round(y))
        col.append(arr[yi, xi] - med if (0 <= xi < W and 0 <= yi < Hh) else np.nan)
    cols.append(col)
    sidx.append(s)
gstrip = np.array(cols).T
sidx = np.array(sidx)
gt_s = (sidx - sidx[0]) / rate
PB = 10
nperp = gstrip.shape[0] // PB
TB = 2.0
tbins = np.arange(gt_s.min(), gt_s.max() + TB, TB)
tcen = 0.5 * (tbins[:-1] + tbins[1:])
gb = np.full((nperp, len(tcen)), np.nan)
with np.errstate(invalid="ignore", divide="ignore"):
    sbin = np.nanmean(gstrip[: nperp * PB].reshape(nperp, PB, gstrip.shape[1]), axis=1)
for i in range(len(tcen)):
    m = (gt_s >= tbins[i]) & (gt_s < tbins[i + 1])
    if m.sum():
        with np.errstate(invalid="ignore"):
            gb[:, i] = np.nanmean(sbin[:, m], axis=1)
perp_off = (np.arange(nperp) - nperp / 2) * PB * 0.2

fig, ax = plt.subplots(figsize=(12, 5))
vmin, vmax = np.nanpercentile(gb[np.isfinite(gb)], [5, 99.5])
im = ax.imshow(
    gb,
    aspect="auto",
    origin="lower",
    cmap="gray_r",
    vmin=vmin,
    vmax=vmax,
    extent=[gt_s[0], gt_s[-1], perp_off[0], perp_off[-1]],
)
ax.axhline(0, color="C1", ls=":", lw=1.0, label="TLE-predicted offset (0)")
ax.axhline(
    poff_best * 0.2,
    color="C2",
    ls="--",
    lw=1.0,
    label=f'measured trail offset ({poff_best*0.2:.0f}")',
)
ax.set_xlabel("time within exposure [s]")
ax.set_ylabel("cross-track offset [arcsec]")
ax.set_title(
    f"GOES 19 wide rectified strip (visit {gvisit} det {det}) — bright trail offset from TLE prediction"
)
ax.legend(loc="upper right")
plt.colorbar(im, label="flux [cts]")
plt.tight_layout()
print(
    f'GOES 19 strip binned to {gb.shape} (perp 2" x {TB:.0f}s). The dark horizontal band is the'
)
print(
    f'satellite trail, sitting ~{abs(poff_best*0.2):.0f}" off the TLE prediction -- a clear, real detection.'
)
print(
    "This is exactly the band ASCENT's stacked strip (Section 11b) lacks at ANY offset."
)

### 12a. GOES 19 along-trail light curve — the variation a bright target reveals

GOES 19 is bright enough (SNR ~10⁴) that its trail *is* a per-exposure light curve: position
along the streak = time within the 30 s exposure (~75 px/s, ~13 ms/px).

**Finding the real trail.** The TLE gives an imperfect prediction — for this hit the actual trail
is offset *and* its position angle differs enough that extracting along the predicted line clips
only a fragment. So we instead **detect the trail directly from the image**: threshold bright
pixels, label connected components, and keep the long, elongated one (the satellite streak;
stars are compact). We fit a line to those pixels and extract flux along it. (The trail is the
dominant ~350″ elongated component on detector 51; a second, shorter collinear streak nearby is
a *different* GEO satellite in the same belt, not part of GOES 19.)

In [ ]:
from scipy import ndimage

# det chosen in Section 12; arr/var/wcs from that cell. Detect the trail from the image itself.
hot = (arr - med) > 8000
lab, nlab = ndimage.label(hot, structure=np.ones((3, 3)))
# keep long, elongated components (line-like); the satellite trail is the largest such
trail_comps = []
for i in range(1, nlab + 1):
    yy, xx = np.where(lab == i)
    if len(xx) < 100:
        continue
    diag = np.hypot(np.ptp(xx), np.ptp(yy))
    if (
        diag > 200 and diag / np.sqrt(len(xx)) > 4
    ):  # elongated => streak, not a star/galaxy
        trail_comps.append((len(xx), diag, i))
trail_comps.sort(reverse=True)
print(
    f"{len(trail_comps)} elongated trail component(s); using the largest "
    f"({trail_comps[0][1]*0.2:.0f} arcsec long)"
)
big = trail_comps[0][2]
ys_t, xs_t = np.where(lab == big)
cf = np.polyfit(ys_t, xs_t, 1)  # x = cf0*y + cf1  (trail is steep)
for _ in range(3):
    res = xs_t - np.polyval(cf, ys_t)
    sg = 1.4826 * np.median(np.abs(res - np.median(res))) + 1
    keep = np.abs(res - np.median(res)) < 3 * sg
    cf = np.polyfit(ys_t[keep], xs_t[keep], 1)
y0, y1 = ys_t.min(), ys_t.max()
dx, dy = cf[0], 1.0
ln = np.hypot(dx, dy)
nx, ny = -dy / ln, dx / ln  # unit normal in pixels
rate_px = (gr.trail_arcsec / 0.2) / 30.0  # along-trail px per second (sidereal)

fl_g, fv_g, t_g = [], [], []
for y in range(y0, y1 + 1):
    x = np.polyval(cf, y)
    if not (0 <= x < W):
        continue
    core = cv = 0.0
    sky = []
    for t in range(-14, 15):
        xi = int(round(x + nx * t))
        yi = int(round(y + ny * t))
        if 0 <= xi < W and 0 <= yi < Hh:
            val = arr[yi, xi] - med
            if abs(t) <= 4:
                core += val
                cv += var[yi, xi]
            elif 8 <= abs(t) <= 14:
                sky.append(val)
    fl_g.append(core - (np.median(sky) * 9 if sky else 0))
    fv_g.append(cv)
    t_g.append((y - y0) * ln / rate_px)
fl_g = np.array(fl_g)
fv_g = np.array(fv_g)
t_g = np.array(t_g)

f = fl_g[fl_g > 20000]
phot = np.sqrt(fv_g[fl_g > 20000])
chi2 = np.sum((f - f.mean()) ** 2 / fv_g[fl_g > 20000]) / (len(f) - 1)
sm = np.convolve(f, np.ones(20) / 20, "valid")
print(
    f"GOES 19 trail light curve: {t_g.max():.1f}s along the detected trail on detector {det}"
)
print(
    f"  mean {f.mean():.0f} cts, photon error ~{phot.mean():.0f} cts/col ({100*phot.mean()/f.mean():.2f}%)"
)
print(
    f"  fractional RMS: {100*f.std()/f.mean():.1f}% raw, {100*sm.std()/sm.mean():.1f}% smoothed to ~0.27s"
)
print(
    f"  reduced chi^2 vs constant: {chi2:.0f}  =>  variation is REAL (photon noise tiny)"
)
print(f"  peak-to-peak: {2.5*np.log10(f.max()/f.min()):.2f} mag")

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(t_g, fl_g, lw=0.6, color="C0", alpha=0.85, label="flux / column (~13 ms)")
ax.plot(t_g[: len(sm)], sm, lw=1.6, color="C3", label="smoothed ~0.27 s")
ax.axhline(f.mean(), color="k", lw=0.6, ls="--", label=f"mean {f.mean():.0f} cts")
ax.set_xlabel("time along trail [s]  (within the 30 s exposure)")
ax.set_ylabel("flux / column [cts]")
ax.set_title(
    f"GOES 19 along-trail light curve (det {det}): {100*f.std()/f.mean():.0f}% RMS variation "
    f"(intrinsic; chi2/dof={chi2:.0f})"
)
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
print(
    "\nClear intrinsic brightness modulation (panel/structure glints as the bright satellite is"
)
print(
    "sampled across the exposure) -- far above the ~0.2% photon noise. This is the per-exposure"
)
print(
    "variability diagnostic ASCENT was too faint to populate; a Landolt-class controlled source"
)
print("would instead be FLAT here -- the calibration ideal.")

### 12b. Absolute flux calibration of the light curve — physical units for Landolt

The Section 12a curve is in *instrumental counts*; here we put it in **physical flux density**.

**Anchoring (not the per-row sum).** The per-row trailed extraction in 12a is *not* individually
flux-conserving — a fixed \u00b14 px transverse window summed per detector row overlaps between
adjacent rows along the steep trail, inflating the absolute counts (~3.8\u00d7). The *fractional*
structure (RMS, peak-to-peak) is immune, which is why 12a only reported ratios. So we anchor the
light curve so its time-average equals the **validated full-trail brightness** from Section 12
(SNR 8918, `getPhotoCalib()` \u2192 **AB 11.67 = 77.8 mJy** in i-band), and carry the measured
fractional modulation on top.

**Result.** GOES\u201119 sits at **~78 mJy** (7.8\u00d710\u207b\u00b2\u2078 W m\u207b\u00b2 Hz\u207b\u00b9; ~6\u00d710\u207b\u00b9\u2074 W m\u207b\u00b2 in-band).
Its **1.8 % smoothed RMS = \u00b11.4 mJy (\u00b10.02 mag)**; the raw 13 ms RMS is **\u00b15.7 mJy (7.3 %)**.
A Landolt-class controlled source should be **flat** here at the mJy level \u2014 that is the
calibration ideal, and 78 mJy at AB 11.67 is the concrete anchor for scaling required laser output:
*f_\u03bd(AB) = 78 mJy \u00d7 10^[-(AB-11.67)/2.5]*.

**GOES‑19 brightness in physical units (i-band):**

| Quantity | Value |
|---|---|
| Mean brightness | **77.8 mJy** (AB 11.67) = 7.8×10⁻²⁸ W m⁻² Hz⁻¹ |
| In-band energy flux | 6.1×10⁻¹⁴ W m⁻² |
| Photon flux | 2.3×10⁵ ph s⁻¹ m⁻² (8.1×10⁶ ph/s into Rubin's ~35 m²) |
| 1.8 % smoothed RMS (~0.27 s) | ±1.4 mJy (±0.020 mag) |
| 7.3 % raw RMS (13 ms) | ±5.7 mJy (±0.080 mag) |
| Peak-to-peak | 2.20 mag (glints) |

**Landolt anchor:** *f_ν(AB) = 78 mJy × 10^[-(AB-11.67)/2.5]* — the conversion from a target on-sky magnitude to required source flux.

In [ ]:
# Absolute calibration of the 12a light curve -> physical flux density (mJy / SI).
# Uses state from Sections 12 (calib, core_sum) and 12a (fl_g, fv_g, t_g).
fnu_full_mJy = (
    calib.instFluxToNanojansky(core_sum) * 1e-6
)  # validated full-trail brightness [mJy]
sel = fl_g > 20000
anchor = fnu_full_mJy / fl_g[sel].mean()  # mJy per count, pinned to the SNR-8918 scale
fnu_mJy = fl_g * anchor  # instantaneous static-equivalent flux density
fnu_err = np.sqrt(fv_g) * anchor
fphys = fnu_mJy[sel]
sm_p = np.convolve(fphys, np.ones(20) / 20, "valid")

# i-band passband for energy/photon flux
LAM, C_LIGHT, H = 0.753e-6, 2.998e8, 6.626e-34
NU, DNU = C_LIGHT / LAM, C_LIGHT * 0.149e-6 / 0.753e-6**2  # nu_eff, i-band width
to_AB = lambda mjy: -2.5 * np.log10(np.asarray(mjy) * 1e-3 / 3631)
fnu_SI = fphys.mean() * 1e-29  # W m^-2 Hz^-1  (1 mJy = 1e-29)

print(
    f"anchor: full-trail f_nu = {fnu_full_mJy:.1f} mJy (AB {to_AB(fnu_full_mJy):.2f})  [validated SNR-8918 scale]"
)
print(f"GOES 19 calibrated light curve (det {det}, i-band):")
print(f"  mean        : {fphys.mean():.1f} mJy  (AB {to_AB(fphys.mean()):.2f})")
print(
    f"  raw RMS     : {fphys.std():.2f} mJy = {100*fphys.std()/fphys.mean():.1f}%  ({1.0857*fphys.std()/fphys.mean():.3f} mag, 13 ms)"
)
print(
    f"  smoothed RMS: {sm_p.std():.2f} mJy = {100*sm_p.std()/sm_p.mean():.1f}%  ({1.0857*sm_p.std()/sm_p.mean():.3f} mag, ~0.27 s)"
)
print(
    f"  peak-to-peak: {2.5*np.log10(fphys.max()/fphys.min()):.2f} mag (endpoints include edge columns)"
)
print(
    f"  SI          : f_nu={fnu_SI:.2e} W/m^2/Hz; in-band f_nu*dnu={fnu_SI*DNU:.2e} W/m^2; "
    f"photon flux={fnu_SI*DNU/(H*NU):.2e} ph/s/m^2 (= {fnu_SI*DNU/(H*NU)*35:.1e} ph/s into Rubin's ~35 m^2)"
)
print(
    f"  Landolt scaling: f_nu(AB) = {fnu_full_mJy:.0f} mJy * 10^[-(AB-{to_AB(fnu_full_mJy):.2f})/2.5]"
)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(
    t_g, fnu_mJy, lw=0.6, color="C0", alpha=0.85, label="flux density / column (~13 ms)"
)
ax.plot(t_g[sel][: len(sm_p)], sm_p, lw=1.6, color="C3", label="smoothed ~0.27 s")
ax.axhline(
    fphys.mean(),
    color="k",
    lw=0.6,
    ls="--",
    label=f"mean {fphys.mean():.0f} mJy (AB {to_AB(fphys.mean()):.2f})",
)
secax = ax.secondary_yaxis(
    "right",
    functions=(
        lambda y: to_AB(np.clip(y, 1e-3, None)),
        lambda m: 3631e3 * 10 ** (-np.asarray(m) / 2.5),
    ),
)
secax.set_ylabel("AB mag (i)")
ax.set_ylim(0, None)
ax.set_xlabel("time along trail [s]  (within the 30 s exposure)")
ax.set_ylabel("flux density [mJy]")
ax.set_title(
    f"GOES 19 calibrated along-trail light curve (det {det}): {fphys.mean():.0f} mJy mean, "
    f"{100*fphys.std()/fphys.mean():.0f}% RMS  (anchored to full-trail AB {to_AB(fnu_full_mJy):.2f})"
)
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
print(
    "\nGOES 19 brightness in physical units: ~78 mJy with intrinsic glint modulation. A Landolt-class"
)
print(
    "controlled source would be FLAT here at the mJy level -- 78 mJy / AB 11.67 is the anchor for"
)
print("converting a target on-sky magnitude into required laser output flux.")

## 13. Second epoch — ASCENT's 2026 FoV crossings (`main` / prompt)

Sections 8–11 measured only the **July 2025** window because that is all DP2 covers. But the
Section 7b cross-match found ASCENT inside the LSSTCam FoV on **four more nights in 2026**
(7 exposures). DP2 lacks them, but the **`main` repo** has them as Prompt-Processing single-frame
products (`preliminary_visit_image`) in per-night `LSSTCam/runs/prompt/<night>/SingleFrame/…`
collections. Six of the seven are processed (the 2026-01-18 visit has only raws), spanning
**two bands (z, i) and three nights** — a genuine second-epoch test of the non-detection.

This section re-runs the **same star-masked trail photometry** (Section 8b machinery: per-column
±6 px swath, DETECTED|CR|SAT mask-plane rejection) and the **same stacking test** (Section 11) on
those exposures, now pointed at `Butler("main")`.

**Two caveats on what this proves.** (1) These are 15–30 s single-frame images taken at **low
altitude** (alt 25–30°, airmass ~2), so the limits are **shallow (~AB 18–20)** — they rule out a
*bright* trail but don't reach the depth where ASCENT is expected. (2) Extraction follows the
**TLE-predicted** trail, which carries the deg-scale GEO cross-track error Section 12 exposed for
GOES-19 (~90″); for a faint non-detection we cannot recenter, so a real trail offset by the TLE
error could fall partly outside the swath. These are limits *along the prediction*, with that
caveat — consistent with, but not independently deeper than, the July stack.

In [ ]:
# Section 13 is self-contained: it uses the `main` repo (not the DP2 Butler from Section 8).
import numpy as np, pandas as pd
import lsst.daf.butler as _dafb, lsst.geom as _geom, lsst.sphgeom as _sph

_main = _dafb.Butler("main")
_mreg = _main.registry
IMG_DT_2 = "preliminary_visit_image"

# per-visit prompt SingleFrame collections in `main` (located via registry search)
COLL_2026 = {
    2026042600279: "LSSTCam/runs/prompt/20260426/SingleFrame/pipelines-d5d9eb5-config-8f017ea",
    2026042600292: "LSSTCam/runs/prompt/20260426/SingleFrame/pipelines-d5d9eb5-config-8f017ea",
    2026050100558: "LSSTCam/runs/prompt/20260501/SingleFrame/pipelines-4e2d5a4-config-8f017ea",
    2026052400724: "LSSTCam/runs/prompt/20260524/SingleFrame/pipelines-5042968-config-8f017ea",
    2026052400730: "LSSTCam/runs/prompt/20260524/SingleFrame/pipelines-5042968-config-8f017ea",
    2026052400743: "LSSTCam/runs/prompt/20260524/SingleFrame/pipelines-5042968-config-8f017ea",
}
_REJECT = ("DETECTED", "CR", "SAT")


def _grow(flagged, wings=4):
    m = flagged.copy()
    for i in np.where(flagged)[0]:
        m[max(0, i - wings) : i + wings + 1] = True
    return m


def dets_on_trail_main(visit, coll, ra0, dec0, ra1, dec1, n=40):
    have = {
        ref.dataId["detector"]
        for ref in _mreg.queryDatasets(
            IMG_DT_2, where=f"visit={visit}", collections=coll, instrument="LSSTCam"
        )
    }
    regs = {
        rec.detector: rec.region
        for rec in _mreg.queryDimensionRecords(
            "visit_detector_region", where=f"visit={visit}", instrument="LSSTCam"
        )
        if rec.region
    }
    out = set()
    for ra, dec in zip(np.linspace(ra0, ra1, n), np.linspace(dec0, dec1, n)):
        uv = _sph.UnitVector3d(_sph.LonLat.fromDegrees(ra, dec))
        for det, rg in regs.items():
            if det in have and rg.contains(uv):
                out.add(det)
    return sorted(out)


def along_trail_main(visit, coll, det, r, swath=6):
    exp = _main.get(
        IMG_DT_2, visit=visit, detector=det, collections=coll, instrument="LSSTCam"
    )
    wcs, calib = exp.getWcs(), exp.getPhotoCalib()
    if wcs is None or calib is None:
        return None
    arr = exp.image.array.astype(float)
    var = exp.variance.array
    H, W = arr.shape
    mk = exp.mask.array
    mpd = exp.mask.getMaskPlaneDict()
    rb = 0
    for p in _REJECT:
        if p in mpd:
            rb |= 1 << mpd[p]
    med = np.nanmedian(arr)
    p0 = wcs.skyToPixel(
        _geom.SpherePoint(r.trail_ra0 * _geom.degrees, r.trail_dec0 * _geom.degrees)
    )
    p1 = wcs.skyToPixel(
        _geom.SpherePoint(r.trail_ra1 * _geom.degrees, r.trail_dec1 * _geom.degrees)
    )
    d = np.array([p1.x - p0.x, p1.y - p0.y])
    L = np.hypot(*d)
    if L < 1:
        return None
    u = d / L
    nn = np.array([-u[1], u[0]])
    fl, fv, fg = [], [], []
    for s in range(int(L)):
        base = np.array([p0.x, p0.y]) + u * s
        cx, cy = int(round(base[0])), int(round(base[1]))
        if not (0 <= cx < W and 0 <= cy < H):
            continue
        fs = vs = 0.0
        flagged = False
        for t in range(-swath, swath + 1):
            x, y = base + nn * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < H:
                fs += arr[yi, xi] - med
                vs += var[yi, xi]
                if mk[yi, xi] & rb:
                    flagged = True
        fl.append(fs)
        fv.append(vs)
        fg.append(flagged)
    return np.array(fl), np.array(fv), np.array(fg, dtype=bool), calib


def _abmag(cts, calib):
    return (
        -2.5 * np.log10(calib.instFluxToNanojansky(cts) * 1e-9 / 3631)
        if cts > 0
        else np.nan
    )


# per-exposure star-masked photometry of the six processed 2026 hits
hits_all = pd.read_parquet("../data/ascent_fov_hits.parquet")
sec2 = hits_all[hits_all.exposure_id.isin(COLL_2026) & hits_all.in_fov].copy()
SNR_CLEAN = 5.0
rows = []
for _, r in sec2.iterrows():
    v = int(r.exposure_id)
    coll = COLL_2026[v]
    dets = dets_on_trail_main(
        v, coll, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
    )
    P = [along_trail_main(v, coll, det, r) for det in dets]
    P = [x for x in P if x]
    if not P:
        rows.append(
            dict(
                night=r.night,
                visit=v,
                band=r.band,
                sep_deg=round(float(r.sep_deg), 2),
                n_det=len(dets),
                clean_snr=np.nan,
                clean_ul_mag=np.nan,
            )
        )
        continue
    flux = np.concatenate([x[0] for x in P])
    fvar = np.concatenate([x[1] for x in P])
    clean = ~_grow(np.concatenate([x[2] for x in P]), wings=4)
    calib = P[0][3]
    cln = flux[clean].sum()
    cvar = fvar[clean].sum()
    csnr = cln / np.sqrt(cvar) if cvar > 0 else np.nan
    rows.append(
        dict(
            night=r.night,
            visit=v,
            band=r.band,
            sep_deg=round(float(r.sep_deg), 2),
            n_det=len(dets),
            clean_snr=csnr,
            clean_ul_mag=_abmag(3 * np.sqrt(cvar), calib) if cvar > 0 else np.nan,
        )
    )
bright_2026 = pd.DataFrame(rows)
bright_2026.to_parquet("../data/ascent_brightness_clean_2026.parquet")
ndet = int((bright_2026.clean_snr > SNR_CLEAN).sum())
print(
    f"{ndet}/{len(bright_2026)} second-epoch exposures detected at SNR>{SNR_CLEAN:.0f} (per-exposure)."
)
print("Per-exposure star-masked photometry (TLE-predicted trail, +/-6px swath):")
display(bright_2026.round(2))

In [ ]:
# Stack the three 2026-05-24 i-band passes (same machinery as Section 11) for a deeper limit.
import matplotlib.pyplot as plt

COLL_524 = "LSSTCam/runs/prompt/20260524/SingleFrame/pipelines-5042968-config-8f017ea"
HALF_T2 = 10
_wing2 = np.abs(np.arange(-HALF_T2, HALF_T2 + 1)) > 6


def transverse_main(visit, coll, det, r):
    exp = _main.get(
        IMG_DT_2, visit=visit, detector=det, collections=coll, instrument="LSSTCam"
    )
    wcs, calib = exp.getWcs(), exp.getPhotoCalib()
    if wcs is None or calib is None:
        return None
    arr = exp.image.array.astype(float)
    var = exp.variance.array
    Hh, W = arr.shape
    mk = exp.mask.array
    mpd = exp.mask.getMaskPlaneDict()
    rb = 0
    for p in _REJECT:
        if p in mpd:
            rb |= 1 << mpd[p]
    p0 = wcs.skyToPixel(
        _geom.SpherePoint(r.trail_ra0 * _geom.degrees, r.trail_dec0 * _geom.degrees)
    )
    p1 = wcs.skyToPixel(
        _geom.SpherePoint(r.trail_ra1 * _geom.degrees, r.trail_dec1 * _geom.degrees)
    )
    d = np.array([p1.x - p0.x, p1.y - p0.y])
    L = np.hypot(*d)
    if L < 1:
        return None
    u = d / L
    nn = np.array([-u[1], u[0]])
    profs, vrs, flags = [], [], []
    for s in range(int(L)):
        base = np.array([p0.x, p0.y]) + u * s
        cx, cy = int(round(base[0])), int(round(base[1]))
        if not (0 <= cx < W and 0 <= cy < Hh):
            continue
        pr, pv, flag = [], [], False
        for t in range(-HALF_T2, HALF_T2 + 1):
            x, y = base + nn * t
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < W and 0 <= yi < Hh:
                pr.append(arr[yi, xi])
                pv.append(var[yi, xi])
                if mk[yi, xi] & rb:
                    flag = True
            else:
                pr.append(np.nan)
                pv.append(np.nan)
        profs.append(pr)
        vrs.append(pv)
        flags.append(flag)
    profs = np.array(profs)
    vrs = np.array(vrs)
    flags = np.array(flags, dtype=bool)
    clean = ~_grow(flags, wings=4)
    profs, vrs = profs[clean], vrs[clean]
    good = np.isfinite(profs).all(axis=1)
    profs, vrs = profs[good], vrs[good]
    if len(profs) == 0:
        return None
    profs = profs - np.median(profs[:, _wing2], axis=1, keepdims=True)
    return profs, vrs


may24 = hits_all[(hits_all.night == "2026-05-24") & hits_all.in_fov]
allP, allV, calib0 = [], [], None
for _, r in may24.iterrows():
    v = int(r.exposure_id)
    dets = dets_on_trail_main(
        v, COLL_524, r.trail_ra0, r.trail_dec0, r.trail_ra1, r.trail_dec1
    )
    for det in dets:
        out = transverse_main(v, COLL_524, det, r)
        if out:
            allP.append(out[0])
            allV.append(out[1])
    if calib0 is None and dets:
        calib0 = _main.get(
            IMG_DT_2 + ".photoCalib",
            visit=v,
            detector=dets[0],
            collections=COLL_524,
            instrument="LSSTCam",
        )
Pstack2 = np.vstack(allP)
Vstack2 = np.vstack(allV)

w = 1.0 / Vstack2


def _coadd(M):
    return np.nansum(M * w, axis=0) / np.nansum(w, axis=0)


offs = np.arange(-HALF_T2, HALF_T2 + 1) * 0.2
err = np.sqrt(1.0 / np.nansum(w, axis=0))
coreS = slice(HALF_T2 - 2, HALF_T2 + 3)
real = _coadd(Pstack2)
real_sum = np.nansum(real[coreS])
real_err = np.sqrt(np.nansum(err[coreS] ** 2))

inj_rows = []
for inj in [2.0, 5.0, 10.0]:
    g = np.exp(-0.5 * (np.arange(-HALF_T2, HALF_T2 + 1) / 1.5) ** 2)
    g = g / g.sum() * inj
    rec = np.nansum(_coadd(Pstack2 + g[None, :])[coreS])
    inj_rows.append((inj, rec, rec / real_err))
recov = (inj_rows[-1][1] - real_sum) / 10.0
ul = 3 * real_err / max(recov, 0.5)
trail_px = may24.trail_arcsec.mean() / 0.2
ul_mag = _abmag(ul * trail_px, calib0)

print(f"N = {len(Pstack2)} clean profiles (3 May-24 i-band passes)")
print(
    f"REAL offset-0 stack: central sum {real_sum:+.2f} +/- {real_err:.2f}  (SNR {real_sum/real_err:+.1f})"
)
for inj, rec, snr in inj_rows:
    print(
        f"  inject {inj:4.1f} cts/col -> recovered {rec-real_sum:+.2f} (SNR {snr:+.1f})"
    )
print(f"  recovered fraction ~ {recov:.2f}  (stacker linear/unbiased)")
print(
    f"3-sigma per-column UL {ul:.1f} cts -> over {trail_px:.0f}px trail -> AB {ul_mag:.2f}  (~1 mag deeper than per-exposure)"
)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.errorbar(
    offs, real, yerr=err, fmt="o-", label=f"real stack (May-24 i, N={len(Pstack2)})"
)
g10 = np.exp(-0.5 * (np.arange(-HALF_T2, HALF_T2 + 1) / 1.5) ** 2)
g10 = g10 / g10.sum() * 10
ax.plot(offs, real + g10, "r--", alpha=0.6, label="if inj=10 cts/col present")
ax.axvline(0, color="0.7", lw=0.8)
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("transverse offset [arcsec]")
ax.set_ylabel("coadded flux/px [cts]")
ax.set_title(
    f"ASCENT 2026-05-24 stacked transverse profile (N={len(Pstack2)}) — no peak at 0"
)
ax.legend()
plt.tight_layout()
print(
    f"\nCONCLUSION: second epoch confirms the July 2025 result. Across 6 exposures / 2 bands / 3"
)
print(
    f"nights ASCENT is undetected; the deeper May-24 i-band stack gives 3-sigma AB ~{ul_mag:.1f}."
)
print(
    "Injections recover cleanly, so the null is physical, not a pipeline failure -- a 12U CubeSat"
)
print("at GEO stays well below Rubin's single-visit reach in both epochs.")